# QF4211 Project Research Notebook


**Required Packages**
Install these packages in the Python environment used to run this notebook before execution:
- `numpy`
- `pandas`
- `matplotlib`
- `joblib`
- `scikit-learn`
- `xgboost`
- `lightgbm`
- `tensorflow`

The notebook assumes these packages are available and imports them directly.


## 1. Scope and Experiment Design
This notebook is the single authoritative pipeline for the project.

It does four things in one place:
- builds the hourly research dataset and paper-aligned feature sets,
- trains all 9 strategy families across 4 feature sets, 3 strategy modes, 2 cost regimes, and 4 assets,
- exports standardized trading logs and paper-style summary tables,
- runs grouped bootstrap tests plus usefulness checks against zero and buy-and-hold.

The older one-off benchmark/model blocks were removed so the notebook now follows one consistent experimental path.


In [129]:
import os
import json
import numpy as np
import pandas as pd
import joblib
import xgboost as xgb
import lightgbm as lgb
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from pathlib import Path
from itertools import product
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-whitegrid')


In [130]:
# -- Execution switches ------------------------------------
FORCE_DATA_PREP = True  # regenerate engineered features and chronological splits from raw CSV files
RUN_UNIFIED_GRID = True   # run the main experiment grid that drives all downstream tables and plots
PRINT_DEBUG_MESSAGES = True
PRINT_MODEL_TRAINING_MESSAGES = True
LOGISTIC_MODEL_VERBOSE = 1 if PRINT_MODEL_TRAINING_MESSAGES else 0
XGBOOST_MODEL_VERBOSITY = 2 if PRINT_MODEL_TRAINING_MESSAGES else 0
LIGHTGBM_MODEL_VERBOSITY = 1 if PRINT_MODEL_TRAINING_MESSAGES else -1
SEQUENCE_MODEL_FIT_VERBOSE = 1 if PRINT_MODEL_TRAINING_MESSAGES else 0

def debug_print(*args, **kwargs):
    if PRINT_DEBUG_MESSAGES:
        print('[DEBUG]', *args, **kwargs)


# Cross-platform worker detection for parallel sections.
# macOS/Linux usually expose CPU count directly through os.cpu_count().
# On Windows, multiprocessing.cpu_count() is kept as a fallback if os.cpu_count() is unavailable.
def detect_parallel_workers(reserve=1):
    cpu_count = os.cpu_count()
    if cpu_count is None:
        try:
            import multiprocessing as mp
            cpu_count = mp.cpu_count()
        except Exception:
            cpu_count = 1
    cpu_count = max(1, int(cpu_count))
    return max(1, cpu_count - reserve)


AVAILABLE_WORKERS = detect_parallel_workers(reserve=1)
UNIFIED_PARALLEL_JOBS = max(1, min(4, AVAILABLE_WORKERS))
RULE_PARALLEL_JOBS = max(1, min(4, AVAILABLE_WORKERS))
BOOTSTRAP_PARALLEL_JOBS = max(1, min(8, AVAILABLE_WORKERS))
TREE_MODEL_N_JOBS = 1 if UNIFIED_PARALLEL_JOBS > 1 else max(1, min(8, AVAILABLE_WORKERS))

# -- Paths --------------------------------------------------
DATA_RAW_DIR = Path('data/raw')
DATA_FEAT_DIR = Path('data/engineered_features')
DATA_SPLIT_DIR = Path('data/split_data')
RESULT_DIR = Path('result')
MODELS_DIR = Path('models')
DATA_LOG_DIR = Path('data/trading_logs')

for d in [
    DATA_FEAT_DIR,
    DATA_SPLIT_DIR,
    MODELS_DIR,
    RESULT_DIR / 'buy_and_hold',
    RESULT_DIR / 'traditional_models',
    RESULT_DIR / 'logistic_regression',
    RESULT_DIR / 'xgboost',
    RESULT_DIR / 'lightgbm',
    RESULT_DIR / 'mlp',
    RESULT_DIR / 'lstm',
    RESULT_DIR / 'cnn',
    RESULT_DIR / 'gru',
    RESULT_DIR / 'paper_tables',
    RESULT_DIR / 'stat_tests',
    DATA_LOG_DIR,
]:
    d.mkdir(parents=True, exist_ok=True)

# -- Experiment constants ----------------------------------
ASSETS = ['BTC', 'ETH', 'XRP', 'SOL']
TRANSACTION_COST = 0.00035
PERIODS_PER_YEAR = 8760
EPS = 1e-12
LOOKBACK_HOURS = 24
LONG_T_GRID = np.arange(0.50, 0.75, 0.01)
SHORT_T_GRID = np.arange(0.50, 0.75, 0.01)

INPUT_FILES = {
    'BTC': str(DATA_RAW_DIR / 'BTCUSDT_1h.csv'),
    'ETH': str(DATA_RAW_DIR / 'ETHUSDT_1h.csv'),
    'XRP': str(DATA_RAW_DIR / 'XRPUSDT_1h.csv'),
    'SOL': str(DATA_RAW_DIR / 'SOLUSDT_1h.csv'),
}

TARGET_LONG_COL = 'label_up'
TARGET_SHORT_COL = 'label_down'
RETURN_COL = 'fwd_ret_open_1h'
LOG_RETURN_COL = 'fwd_log_ret_open_1h'

PAPER_CYCLICAL_FEATURES = ['hour_sin', 'hour_cos']
PAPER_CANDLE_FEATURES = ['close', 'candle_body', 'upper_shadow', 'lower_shadow']
PAPER_OHLC_FEATURES = ['open', 'high', 'low', 'close']
PAPER_TECH_IND_FEATURES = [
    'sma_15', 'sma_20', 'sma_25', 'sma_30',
    'rsi_15', 'rsi_20', 'rsi_25', 'rsi_30',
    'macd_line', 'macd_hist', 'wr_14', 'so_14', 'mfi_14',
]

OHLC_RAW_FEATURES = PAPER_OHLC_FEATURES + ['number_of_trades', 'volume'] + PAPER_CYCLICAL_FEATURES
CANDLE_RAW_FEATURES = PAPER_CANDLE_FEATURES + ['number_of_trades', 'volume'] + PAPER_CYCLICAL_FEATURES
TECH_IND_FEATURES = PAPER_TECH_IND_FEATURES
FEATURE_SET_DEFS = {
    'ohlc_raw': OHLC_RAW_FEATURES,
    'candle_raw': CANDLE_RAW_FEATURES,
    'ohlc_extended': OHLC_RAW_FEATURES + TECH_IND_FEATURES,
    'candle_extended': CANDLE_RAW_FEATURES + TECH_IND_FEATURES,
}

EXTRA_FEATURE_COLS = [
    'quote_asset_volume', 'taker_buy_base_asset_volume', 'taker_buy_quote_asset_volume',
    'ret_1h_close', 'log_ret_1h_close', 'ret_3h', 'log_ret_3h', 'ret_6h', 'log_ret_6h',
    'ret_12h', 'log_ret_12h', 'ret_24h', 'log_ret_24h', 'ret_48h', 'log_ret_48h',
    'ret_72h', 'log_ret_72h', 'body_pct', 'body_abs_pct', 'range_pct', 'body_to_range',
    'close_loc', 'bar_up', 'bar_down', 'gap_open_prev_close', 'close_to_sma_6',
    'close_to_sma_12', 'close_to_sma_24', 'close_to_sma_48', 'close_to_sma_72',
    'close_to_ema_6', 'close_to_ema_12', 'close_to_ema_24', 'close_to_ema_48',
    'close_to_ema_72', 'sma_cross_6_24', 'sma_cross_12_48', 'ema_cross_12_24',
    'vol_6h', 'vol_24h', 'vol_72h', 'downside_vol_24h', 'downside_vol_72h',
    'atr_14_pct', 'macd_signal', 'macd_signal_pct', 'macd_line_pct', 'macd_hist_pct',
    'log_volume', 'log_quote_volume', 'log_trades', 'vol_z_24', 'quote_vol_z_24',
    'trades_z_24', 'vol_ratio_24', 'quote_vol_ratio_24', 'taker_buy_base_share',
    'taker_buy_quote_share', 'order_flow_imbalance', 'dow_sin', 'dow_cos',
]
FEATURE_COLS = sorted(set(sum(FEATURE_SET_DEFS.values(), [])) | set(EXTRA_FEATURE_COLS))
FULL_EXTENDED_FEATURES = sorted(set(FEATURE_COLS))

UNIFIED_MODELS = ['Logistic Regression', 'XGBoost', 'LightGBM', 'MLP', 'LSTM', 'CNN', 'GRU', 'SMA', 'RSI']
ML_MODELS = ['Logistic Regression', 'XGBoost', 'LightGBM', 'MLP', 'LSTM', 'CNN', 'GRU']
TABULAR_ML_MODELS = ['Logistic Regression', 'XGBoost', 'LightGBM']
SEQUENCE_ML_MODELS = ['MLP', 'LSTM', 'CNN', 'GRU']
RULE_MODELS = ['SMA', 'RSI']
UNIFIED_FEATURE_SETS = ['ohlc_raw', 'candle_raw', 'ohlc_extended', 'candle_extended']
UNIFIED_STRATEGY_MODES = ['long_only', 'short_only', 'long_short']
COST_REGIMES = {'with_cost': TRANSACTION_COST, 'no_cost': 0.0}

SMA_SHORT_WINDOWS = [5, 10, 20, 30]
SMA_LONG_WINDOWS = [50, 100, 150, 200]
RSI_WINDOWS = [7, 14, 21]

MIN_TRADES_VALID = 20
MIN_SIGNAL_RATIO = 0.002
MIN_PRECISION_RECALL = 0.01


## 2. Data Preparation and Paper-Aligned Feature Engineering
The feature-engineering stage converts raw Binance hourly bars into the four feature families used in the main experiment grid.

The main comparison uses:
- `ohlc_raw`
- `candle_raw`
- `ohlc_extended`
- `candle_extended`

Extra engineered columns are still kept for diagnostics, but they are not part of the paper-aligned four-set comparison.


In [131]:
# -- Indicator helpers ----------------------------------
def compute_rsi(close: pd.Series, window: int = 14) -> pd.Series:
    delta = close.diff()
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)
    avg_gain = gain.ewm(alpha=1 / window, adjust=False, min_periods=window).mean()
    avg_loss = loss.ewm(alpha=1 / window, adjust=False, min_periods=window).mean()
    rs = avg_gain / (avg_loss + EPS)
    return 100 - (100 / (1 + rs))


def compute_macd(close, fast=12, slow=26, signal=9):
    ema_f = close.ewm(span=fast, adjust=False, min_periods=fast).mean()
    ema_s = close.ewm(span=slow, adjust=False, min_periods=slow).mean()
    line = ema_f - ema_s
    sig = line.ewm(span=signal, adjust=False, min_periods=signal).mean()
    return line, sig, line - sig


def compute_atr(high, low, close, window=14):
    pc = close.shift(1)
    tr = pd.concat([high - low, (high - pc).abs(), (low - pc).abs()], axis=1).max(axis=1)
    return tr.rolling(window, min_periods=window).mean()


def compute_williams_r(high, low, close, window=14):
    hh = high.rolling(window, min_periods=window).max()
    ll = low.rolling(window, min_periods=window).min()
    return -100 * (hh - close) / (hh - ll + EPS)


def compute_stochastic_oscillator(high, low, close, window=14):
    hh = high.rolling(window, min_periods=window).max()
    ll = low.rolling(window, min_periods=window).min()
    k = 100 * (close - ll) / (hh - ll + EPS)
    d = k.rolling(3, min_periods=3).mean()
    return k - d


def compute_mfi(high, low, close, volume, window=14):
    tp = (high + low + close) / 3
    rmf = tp * volume
    tp_diff = tp.diff()
    pos = rmf.where(tp_diff > 0, 0.0)
    neg = rmf.where(tp_diff < 0, 0.0).abs()
    pmf = pos.rolling(window, min_periods=window).sum()
    nmf = neg.rolling(window, min_periods=window).sum()
    mfr = pmf / (nmf + EPS)
    return 100 - 100 / (1 + mfr)


def rolling_zscore(series, window):
    m = series.rolling(window, min_periods=window).mean()
    s = series.rolling(window, min_periods=window).std()
    return (series - m) / (s + EPS)


In [132]:
# -- Raw-data loading -----------------------------------
def load_and_standardize(file_path: str, asset_name: str) -> pd.DataFrame:
    df = pd.read_csv(file_path)
    keep = [
        'open_time', 'open', 'high', 'low', 'close', 'volume',
        'quote_asset_volume', 'number_of_trades',
        'taker_buy_base_asset_volume', 'taker_buy_quote_asset_volume',
    ]
    df = df[keep].copy()
    df['open_time'] = pd.to_numeric(df['open_time'], errors='coerce')
    dt_ms = pd.to_datetime(df['open_time'], unit='ms', utc=True, errors='coerce')
    dt_us = pd.to_datetime(df['open_time'], unit='us', utc=True, errors='coerce')
    df['datetime'] = dt_ms.fillna(dt_us)
    df = df.dropna(subset=['datetime']).sort_values('datetime').reset_index(drop=True)
    for c in [
        'open', 'high', 'low', 'close', 'volume', 'quote_asset_volume',
        'number_of_trades', 'taker_buy_base_asset_volume', 'taker_buy_quote_asset_volume',
    ]:
        df[c] = pd.to_numeric(df[c], errors='coerce')
    df = df.drop_duplicates('datetime').dropna(subset=['open', 'high', 'low', 'close'])
    df = df[(df['open'] > 0) & (df['high'] > 0) & (df['low'] > 0) & (df['close'] > 0)].copy()
    df['asset'] = asset_name
    return df.reset_index(drop=True)


In [133]:
# -- Feature construction -------------------------------
def create_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    out['log_close'] = np.log(out['close'])
    out['ret_1h_close'] = out['close'].pct_change(1)
    out['log_ret_1h_close'] = np.log(out['close'] / out['close'].shift(1))
    for h in [3, 6, 12, 24, 48, 72]:
        out[f'ret_{h}h'] = out['close'].pct_change(h)
        out[f'log_ret_{h}h'] = np.log(out['close'] / out['close'].shift(h))

    max_oc = np.maximum(out['open'], out['close'])
    min_oc = np.minimum(out['open'], out['close'])
    out['range_pct'] = (out['high'] - out['low']) / (out['open'] + EPS)
    out['body_pct'] = (out['close'] - out['open']) / (out['open'] + EPS)
    out['body_abs_pct'] = out['body_pct'].abs()
    out['upper_shadow_pct'] = (out['high'] - max_oc) / (out['open'] + EPS)
    out['lower_shadow_pct'] = (min_oc - out['low']) / (out['open'] + EPS)
    out['body_to_range'] = (out['close'] - out['open']) / ((out['high'] - out['low']) + EPS)
    out['close_loc'] = (out['close'] - out['low']) / ((out['high'] - out['low']) + EPS)
    out['bar_up'] = (out['close'] > out['open']).astype(int)
    out['bar_down'] = (out['close'] < out['open']).astype(int)
    out['gap_open_prev_close'] = out['open'] / out['close'].shift(1) - 1

    out['candle_body'] = (out['close'] - out['open']).abs()
    out['upper_shadow'] = np.where(out['close'] >= out['open'], out['high'] - out['close'], out['high'] - out['open'])
    out['lower_shadow'] = np.where(out['close'] >= out['open'], out['open'] - out['low'], out['close'] - out['low'])

    for w in [6, 12, 24, 48, 72]:
        out[f'sma_{w}'] = out['close'].rolling(w, min_periods=w).mean()
        out[f'ema_{w}'] = out['close'].ewm(span=w, adjust=False, min_periods=w).mean()
        out[f'close_to_sma_{w}'] = out['close'] / (out[f'sma_{w}'] + EPS) - 1
        out[f'close_to_ema_{w}'] = out['close'] / (out[f'ema_{w}'] + EPS) - 1
    out['sma_cross_6_24'] = out['sma_6'] / (out['sma_24'] + EPS) - 1
    out['sma_cross_12_48'] = out['sma_12'] / (out['sma_48'] + EPS) - 1
    out['ema_cross_12_24'] = out['ema_12'] / (out['ema_24'] + EPS) - 1

    for w in [15, 20, 25, 30]:
        out[f'sma_{w}'] = out['close'].rolling(w, min_periods=w).mean()
        out[f'rsi_{w}'] = compute_rsi(out['close'], w)

    for w in [6, 24, 72]:
        out[f'vol_{w}h'] = out['log_ret_1h_close'].rolling(w, min_periods=w).std()
    neg = out['log_ret_1h_close'].where(out['log_ret_1h_close'] < 0, 0)
    for w in [24, 72]:
        out[f'downside_vol_{w}h'] = neg.rolling(w, min_periods=w).std()
    out['atr_14'] = compute_atr(out['high'], out['low'], out['close'], 14)
    out['atr_14_pct'] = out['atr_14'] / (out['close'] + EPS)

    out['rsi_14'] = compute_rsi(out['close'], 14)
    macd_line, macd_signal, macd_hist = compute_macd(out['close'])
    out['macd_line'] = macd_line
    out['macd_signal'] = macd_signal
    out['macd_hist'] = macd_hist
    out['macd_line_pct'] = macd_line / (out['close'] + EPS)
    out['macd_signal_pct'] = macd_signal / (out['close'] + EPS)
    out['macd_hist_pct'] = macd_hist / (out['close'] + EPS)
    out['wr_14'] = compute_williams_r(out['high'], out['low'], out['close'], 14)
    out['so_14'] = compute_stochastic_oscillator(out['high'], out['low'], out['close'], 14)
    out['mfi_14'] = compute_mfi(out['high'], out['low'], out['close'], out['volume'], 14)

    out['log_volume'] = np.log1p(out['volume'])
    out['log_quote_volume'] = np.log1p(out['quote_asset_volume'])
    out['log_trades'] = np.log1p(out['number_of_trades'])
    out['vol_z_24'] = rolling_zscore(out['log_volume'], 24)
    out['quote_vol_z_24'] = rolling_zscore(out['log_quote_volume'], 24)
    out['trades_z_24'] = rolling_zscore(out['log_trades'], 24)
    out['vol_ratio_24'] = out['volume'] / (out['volume'].rolling(24, min_periods=24).mean() + EPS)
    out['quote_vol_ratio_24'] = out['quote_asset_volume'] / (out['quote_asset_volume'].rolling(24, min_periods=24).mean() + EPS)
    out['taker_buy_base_share'] = out['taker_buy_base_asset_volume'] / (out['volume'] + EPS)
    out['taker_buy_quote_share'] = out['taker_buy_quote_asset_volume'] / (out['quote_asset_volume'] + EPS)
    out['order_flow_imbalance'] = 2 * out['taker_buy_base_share'] - 1

    # Defragment before the final time/target columns so pandas does not warn
    # about the repeated feature-column insertions above.
    out = out.copy()

    out['hour'] = out['datetime'].dt.hour
    out['dayofweek'] = out['datetime'].dt.dayofweek
    out['hour_sin'] = np.sin(2 * np.pi * out['hour'] / 24)
    out['hour_cos'] = np.cos(2 * np.pi * out['hour'] / 24)
    out['dow_sin'] = np.sin(2 * np.pi * out['dayofweek'] / 7)
    out['dow_cos'] = np.cos(2 * np.pi * out['dayofweek'] / 7)

    out['entry_open_t1'] = out['open'].shift(-1)
    out['exit_open_t2'] = out['open'].shift(-2)
    out['fwd_ret_open_1h'] = out['exit_open_t2'] / out['entry_open_t1'] - 1
    out['fwd_log_ret_open_1h'] = np.log(out['exit_open_t2'] / out['entry_open_t1'])
    out['label_up'] = (out['fwd_log_ret_open_1h'] > 0).astype(int)
    out['label_down'] = (out['fwd_log_ret_open_1h'] < 0).astype(int)

    out = out.replace([np.inf, -np.inf], np.nan).copy()
    return out


In [134]:
# -- Feature-dataset assembly and execution -------------
def build_feature_dataset(file_path: str, asset_name: str) -> pd.DataFrame:
    raw = load_and_standardize(file_path, asset_name)
    feat = create_features(raw)
    keep_cols = [
        'datetime', 'asset', 'open', 'high', 'low', 'close', 'volume',
        'quote_asset_volume', 'number_of_trades',
        'taker_buy_base_asset_volume', 'taker_buy_quote_asset_volume',
        'entry_open_t1', 'exit_open_t2', RETURN_COL, LOG_RETURN_COL,
        TARGET_LONG_COL, TARGET_SHORT_COL,
        'candle_body', 'upper_shadow', 'lower_shadow',
        'ret_1h_close', 'log_ret_1h_close', 'ret_3h', 'log_ret_3h', 'ret_6h', 'log_ret_6h',
        'ret_12h', 'log_ret_12h', 'ret_24h', 'log_ret_24h', 'ret_48h', 'log_ret_48h',
        'ret_72h', 'log_ret_72h', 'range_pct', 'body_pct', 'body_abs_pct', 'upper_shadow_pct',
        'lower_shadow_pct', 'body_to_range', 'close_loc', 'bar_up', 'bar_down',
        'gap_open_prev_close', 'sma_6', 'sma_12', 'sma_15', 'sma_20', 'sma_24', 'sma_25',
        'sma_30', 'sma_48', 'sma_72', 'ema_6', 'ema_12', 'ema_24', 'ema_48', 'ema_72',
        'close_to_sma_6', 'close_to_sma_12', 'close_to_sma_24', 'close_to_sma_48', 'close_to_sma_72',
        'close_to_ema_6', 'close_to_ema_12', 'close_to_ema_24', 'close_to_ema_48', 'close_to_ema_72',
        'sma_cross_6_24', 'sma_cross_12_48', 'ema_cross_12_24',
        'rsi_14', 'rsi_15', 'rsi_20', 'rsi_25', 'rsi_30',
        'macd_line', 'macd_signal', 'macd_hist', 'macd_line_pct', 'macd_signal_pct', 'macd_hist_pct',
        'wr_14', 'so_14', 'mfi_14', 'vol_6h', 'vol_24h', 'vol_72h', 'downside_vol_24h',
        'downside_vol_72h', 'atr_14', 'atr_14_pct', 'log_volume', 'log_quote_volume', 'log_trades',
        'vol_z_24', 'quote_vol_z_24', 'trades_z_24', 'vol_ratio_24', 'quote_vol_ratio_24',
        'taker_buy_base_share', 'taker_buy_quote_share', 'order_flow_imbalance',
        'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos',
    ]
    feat = feat[keep_cols].dropna().reset_index(drop=True)
    return feat


feat_files_exist = all((DATA_FEAT_DIR / f'{a}_features.csv').exists() for a in ASSETS)

if not FORCE_DATA_PREP and feat_files_exist:
    print('Engineered feature files found - skipping feature engineering.')
    print('Set FORCE_DATA_PREP = True to regenerate.')
else:
    print('Running feature engineering...')
    for asset, file_path in INPUT_FILES.items():
        df_feat = build_feature_dataset(file_path, asset)
        out_path = DATA_FEAT_DIR / f'{asset}_features.csv'
        df_feat.to_csv(out_path, index=False)
        print(f'  {asset}: {len(df_feat):,} rows -> {out_path}')


Running feature engineering...
  BTC: 43,736 rows -> data/engineered_features/BTC_features.csv
  ETH: 43,736 rows -> data/engineered_features/ETH_features.csv
  XRP: 43,736 rows -> data/engineered_features/XRP_features.csv
  SOL: 43,736 rows -> data/engineered_features/SOL_features.csv


## 3. Chronological Train / Validation / Test Split
The split is strictly time ordered:
- 50% training,
- 25% validation,
- 25% test.

No random reshuffling is used. This keeps threshold tuning and out-of-sample evaluation chronologically honest.


In [135]:
def chronological_split(df, train_frac=0.50, val_frac=0.25, test_frac=0.25):
    df = df.sort_values('datetime').reset_index(drop=True)
    n = len(df)
    t1 = int(n * train_frac)
    t2 = int(n * (train_frac + val_frac))
    return df.iloc[:t1].copy(), df.iloc[t1:t2].copy(), df.iloc[t2:].copy()


In [136]:
split_files_exist = all(
    (DATA_SPLIT_DIR / f'{a}_{s}.csv').exists()
    for a in ASSETS for s in ['train', 'val', 'test']
)

if not FORCE_DATA_PREP and split_files_exist:
    print('Split data files found - skipping split.')
    print('Set FORCE_DATA_PREP = True to re-split.')
else:
    print('Running train/val/test split...')
    for asset in ASSETS:
        df = pd.read_csv(DATA_FEAT_DIR / f'{asset}_features.csv')
        df['datetime'] = pd.to_datetime(df['datetime'], utc=True, errors='coerce')
        train_df, val_df, test_df = chronological_split(df)
        for split, sdf in [('train', train_df), ('val', val_df), ('test', test_df)]:
            sdf.to_csv(DATA_SPLIT_DIR / f'{asset}_{split}.csv', index=False)
        print(f'  {asset}: train={len(train_df):,}  val={len(val_df):,}  test={len(test_df):,}')

def load_split_csv(path):
    df = pd.read_csv(path, low_memory=False)
    if 'datetime' in df.columns:
        df['datetime'] = pd.to_datetime(df['datetime'], utc=True, errors='coerce')
    return df


train_data, val_data, test_data = {}, {}, {}
for asset in ASSETS:
    train_data[asset] = load_split_csv(DATA_SPLIT_DIR / f'{asset}_train.csv')
    val_data[asset] = load_split_csv(DATA_SPLIT_DIR / f'{asset}_val.csv')
    test_data[asset] = load_split_csv(DATA_SPLIT_DIR / f'{asset}_test.csv')

print('Data loaded into train_data / val_data / test_data.')


Running train/val/test split...
  BTC: train=21,868  val=10,934  test=10,934
  ETH: train=21,868  val=10,934  test=10,934
  XRP: train=21,868  val=10,934  test=10,934
  SOL: train=21,868  val=10,934  test=10,934
Data loaded into train_data / val_data / test_data.


## 4. Shared Metrics, Threshold Selection, and Bootstrap Helpers
This section centralizes the evaluation logic used throughout the notebook.

Key rules:
- validation thresholds are chosen by precision first,
- thresholds must still satisfy minimum trade-count, signal-ratio, and recall guardrails,
- the paper-style Sharpe in the summary tables is computed from signal returns rather than annualized hourly returns,
- grouped bootstrap tests follow the paper's non-overlapping block idea with block length `floor(n^(1/3))`.


In [137]:
# -- Return and risk summary helpers --------------------
def compute_cumulative_return(returns):
    returns = pd.Series(returns).dropna()
    return (1 + returns).prod() - 1 if len(returns) > 0 else np.nan


def compute_annualized_return(returns, periods_per_year=8760):
    returns = pd.Series(returns).dropna()
    if len(returns) == 0:
        return np.nan
    cum = (1 + returns).prod()
    return cum ** (periods_per_year / len(returns)) - 1 if cum > 0 else np.nan


def compute_annualized_volatility(returns, periods_per_year=8760):
    returns = pd.Series(returns).dropna()
    return returns.std(ddof=1) * np.sqrt(periods_per_year) if len(returns) >= 2 else np.nan


def compute_annualized_sharpe(returns, risk_free_rate=0.0, periods_per_year=8760):
    returns = pd.Series(returns).dropna()
    if len(returns) < 2:
        return np.nan
    excess = returns - risk_free_rate / periods_per_year
    vol = excess.std(ddof=1)
    return excess.mean() / vol * np.sqrt(periods_per_year) if vol > 0 else np.nan


def compute_max_drawdown(returns):
    returns = pd.Series(returns).dropna()
    if len(returns) == 0:
        return np.nan
    eq = (1 + returns).cumprod()
    return (eq / eq.cummax() - 1).min()


def compute_turnover(position):
    position = pd.Series(position).fillna(0)
    return position.diff().abs().fillna(position.iloc[0]).sum()


def compute_num_trades(position):
    position = pd.Series(position).fillna(0)
    return int((position.diff().abs().fillna(position.iloc[0]) > 0).sum())


def compute_signal_returns(backtest_df, return_col='net_return', position_col='position'):
    mask = backtest_df[position_col].fillna(0) != 0
    return pd.Series(backtest_df.loc[mask, return_col]).dropna().reset_index(drop=True)


def compute_avg_signal_return(backtest_df, return_col='net_return', position_col='position'):
    signal_returns = compute_signal_returns(backtest_df, return_col=return_col, position_col=position_col)
    if len(signal_returns) == 0:
        return np.nan
    return compute_cumulative_return(signal_returns) / len(signal_returns)


def compute_signal_return_std(backtest_df, return_col='net_return', position_col='position'):
    signal_returns = compute_signal_returns(backtest_df, return_col=return_col, position_col=position_col)
    return signal_returns.std(ddof=1) if len(signal_returns) >= 2 else np.nan


def compute_paper_sharpe(backtest_df, return_col='net_return', position_col='position'):
    avg_signal_return = compute_avg_signal_return(backtest_df, return_col=return_col, position_col=position_col)
    signal_std = compute_signal_return_std(backtest_df, return_col=return_col, position_col=position_col)
    if pd.isna(avg_signal_return) or pd.isna(signal_std) or signal_std <= 0:
        return np.nan
    return avg_signal_return / signal_std


def summarize_performance(backtest_df, return_col='net_return', position_col='position', periods_per_year=8760, risk_free_rate=0.0):
    returns = backtest_df[return_col].dropna()
    position = backtest_df[position_col]
    return pd.Series({
        'Cumulative Return': compute_cumulative_return(returns),
        'Annualized Return': compute_annualized_return(returns, periods_per_year),
        'Annualized Volatility': compute_annualized_volatility(returns, periods_per_year),
        'Annualized Sharpe': compute_annualized_sharpe(returns, risk_free_rate, periods_per_year),
        'Maximum Drawdown': compute_max_drawdown(returns),
        'Turnover': compute_turnover(position),
        'Number of Trades': compute_num_trades(position),
        'Avg Return': compute_avg_signal_return(backtest_df, return_col=return_col, position_col=position_col),
        'Signal Return Std': compute_signal_return_std(backtest_df, return_col=return_col, position_col=position_col),
    })


In [138]:
# -- Directional metrics and backtest helpers -----------
def _compute_long_metrics(position, raw_return):
    pred = position > 0
    actual = raw_return > 0
    tp = int((pred & actual).sum())
    pp = int(pred.sum())
    ap = int(actual.sum())
    precision = tp / pp if pp > 0 else np.nan
    recall = tp / ap if ap > 0 else np.nan
    return precision, recall


def _compute_short_metrics(position, raw_return):
    pred = position < 0
    actual = raw_return < 0
    tp = int((pred & actual).sum())
    pp = int(pred.sum())
    ap = int(actual.sum())
    precision = tp / pp if pp > 0 else np.nan
    recall = tp / ap if ap > 0 else np.nan
    return precision, recall


def _safe_mean_or_nan(values):
    valid = [v for v in values if pd.notna(v)]
    return float(np.mean(valid)) if valid else np.nan


def compute_directional_metrics(backtest_df, mode='long_only'):
    position = backtest_df['position'].fillna(0).values
    raw_return = backtest_df['raw_return'].fillna(0).values
    if mode == 'long_only':
        precision, recall = _compute_long_metrics(position, raw_return)
    elif mode == 'short_only':
        precision, recall = _compute_short_metrics(position, raw_return)
    elif mode == 'long_short':
        p_long, r_long = _compute_long_metrics(position, raw_return)
        p_short, r_short = _compute_short_metrics(position, raw_return)
        precision = _safe_mean_or_nan([p_long, p_short])
        recall = _safe_mean_or_nan([r_long, r_short])
    else:
        raise ValueError(f'Unsupported mode: {mode}')
    return pd.Series({'Precision': precision, 'Recall': recall})


def _sigmoid_score(score, scale=20.0):
    score = np.asarray(score, dtype=float)
    return 1.0 / (1.0 + np.exp(-np.clip(scale * score, -50, 50)))


def build_positions_from_dual_probabilities(long_prob, short_prob, mode='long_only', long_threshold=0.5, short_threshold=0.5):
    long_prob = np.asarray(long_prob, dtype=float)
    short_prob = np.asarray(short_prob if short_prob is not None else np.zeros_like(long_prob), dtype=float)
    long_threshold = 0.5 if long_threshold is None or pd.isna(long_threshold) else float(long_threshold)
    short_threshold = 0.5 if short_threshold is None or pd.isna(short_threshold) else float(short_threshold)

    if mode == 'long_only':
        return (long_prob >= long_threshold).astype(int)

    if mode == 'short_only':
        pos = np.zeros(len(short_prob), dtype=int)
        pos[short_prob >= short_threshold] = -1
        return pos

    if mode == 'long_short':
        pos = np.zeros(len(long_prob), dtype=int)
        long_active = long_prob >= long_threshold
        short_active = short_prob >= short_threshold
        long_margin = long_prob - long_threshold
        short_margin = short_prob - short_threshold

        pos[long_active & ~short_active] = 1
        pos[short_active & ~long_active] = -1

        both = long_active & short_active
        pos[both & (long_margin > short_margin)] = 1
        pos[both & (short_margin > long_margin)] = -1
        return pos

    raise ValueError(f'Unsupported mode: {mode}')


def run_backtest_from_position(df, position, return_col=RETURN_COL, cost=TRANSACTION_COST):
    out = df.copy().reset_index(drop=True)
    out['position'] = pd.Series(position).astype(float)
    out['raw_return'] = out[return_col].astype(float)
    if LOG_RETURN_COL in out.columns:
        out['raw_log_return'] = out[LOG_RETURN_COL].astype(float)
    out['position_change'] = (out['position'] - out['position'].shift(1).fillna(0)).abs()
    out['transaction_cost'] = cost * out['position_change']
    out['gross_return'] = out['position'] * out['raw_return']
    out['net_return'] = out['gross_return'] - out['transaction_cost']
    out['gross_equity_curve'] = (1 + out['gross_return']).cumprod()
    out['net_equity_curve'] = (1 + out['net_return']).cumprod()
    return out


def summarize_signal_performance(backtest_df, mode='long_only'):
    perf = summarize_performance(backtest_df)
    dirm = compute_directional_metrics(backtest_df, mode=mode)
    return {
        'Sharpe': compute_paper_sharpe(backtest_df),
        'Precision': dirm['Precision'],
        'Recall': dirm['Recall'],
        'Avg Return': perf['Avg Return'],
        'Cumulative Return': perf['Cumulative Return'],
        'Turnover': perf['Turnover'],
        'Number of Trades': perf['Number of Trades'],
        'Maximum Drawdown': perf['Maximum Drawdown'],
        'Annualized Sharpe': perf['Annualized Sharpe'],
    }


In [139]:
# -- Threshold tuning -----------------------------------
def _threshold_score(backtest_df, mode='long_only'):
    metrics = summarize_signal_performance(backtest_df, mode=mode)
    trades = int(metrics['Number of Trades'])
    signal_ratio = float((backtest_df['position'] != 0).mean())
    if trades < MIN_TRADES_VALID or signal_ratio < MIN_SIGNAL_RATIO:
        return -np.inf, {**metrics, 'Trades': trades, 'Signal Ratio': signal_ratio}
    if pd.isna(metrics['Recall']) or metrics['Recall'] < MIN_PRECISION_RECALL:
        return -np.inf, {**metrics, 'Trades': trades, 'Signal Ratio': signal_ratio}
    score = metrics['Precision'] if pd.notna(metrics['Precision']) else -np.inf
    return score, {**metrics, 'Trades': trades, 'Signal Ratio': signal_ratio}


def tune_threshold_precision(long_prob, short_prob, val_df, mode='long_only', long_grid=None, short_grid=None, cost=0.00035):
    long_grid = list(long_grid if long_grid is not None else LONG_T_GRID)
    short_grid = list(short_grid if short_grid is not None else SHORT_T_GRID)
    fallback = {
        'long_threshold': long_grid[len(long_grid) // 2] if len(long_grid) > 0 else 0.5,
        'short_threshold': short_grid[len(short_grid) // 2] if len(short_grid) > 0 else 0.5,
        'Validation Precision': np.nan,
        'Validation Recall': np.nan,
        'Validation Sharpe': np.nan,
        'Validation Avg Return': np.nan,
        'Validation Trades': np.nan,
        'Validation Signal Ratio': np.nan,
        'Threshold Feasible': False,
    }
    if len(val_df) == 0:
        return fallback

    best_key = None
    best = None

    if mode == 'long_only':
        for lt in long_grid:
            pos = build_positions_from_dual_probabilities(long_prob, short_prob, mode='long_only', long_threshold=lt)
            bt = run_backtest_from_position(val_df, pos, return_col=RETURN_COL, cost=cost)
            score, diag = _threshold_score(bt, mode='long_only')
            if not np.isfinite(score):
                continue
            key = (
                score,
                diag['Recall'] if pd.notna(diag['Recall']) else -np.inf,
                diag['Sharpe'] if pd.notna(diag['Sharpe']) else -np.inf,
            )
            if best_key is None or key > best_key:
                best_key = key
                best = {
                    'long_threshold': lt,
                    'short_threshold': None,
                    'Validation Precision': diag['Precision'],
                    'Validation Recall': diag['Recall'],
                    'Validation Sharpe': diag['Sharpe'],
                    'Validation Avg Return': diag['Avg Return'],
                    'Validation Trades': diag['Trades'],
                    'Validation Signal Ratio': diag['Signal Ratio'],
                    'Threshold Feasible': True,
                }
        return best if best is not None else fallback

    if mode == 'short_only':
        for st in short_grid:
            pos = build_positions_from_dual_probabilities(long_prob, short_prob, mode='short_only', short_threshold=st)
            bt = run_backtest_from_position(val_df, pos, return_col=RETURN_COL, cost=cost)
            score, diag = _threshold_score(bt, mode='short_only')
            if not np.isfinite(score):
                continue
            key = (
                score,
                diag['Recall'] if pd.notna(diag['Recall']) else -np.inf,
                diag['Sharpe'] if pd.notna(diag['Sharpe']) else -np.inf,
            )
            if best_key is None or key > best_key:
                best_key = key
                best = {
                    'long_threshold': None,
                    'short_threshold': st,
                    'Validation Precision': diag['Precision'],
                    'Validation Recall': diag['Recall'],
                    'Validation Sharpe': diag['Sharpe'],
                    'Validation Avg Return': diag['Avg Return'],
                    'Validation Trades': diag['Trades'],
                    'Validation Signal Ratio': diag['Signal Ratio'],
                    'Threshold Feasible': True,
                }
        return best if best is not None else fallback

    if mode == 'long_short':
        for lt in long_grid:
            for st in short_grid:
                pos = build_positions_from_dual_probabilities(long_prob, short_prob, mode='long_short', long_threshold=lt, short_threshold=st)
                bt = run_backtest_from_position(val_df, pos, return_col=RETURN_COL, cost=cost)
                score, diag = _threshold_score(bt, mode='long_short')
                if not np.isfinite(score):
                    continue
                key = (
                    score,
                    diag['Recall'] if pd.notna(diag['Recall']) else -np.inf,
                    diag['Sharpe'] if pd.notna(diag['Sharpe']) else -np.inf,
                )
                if best_key is None or key > best_key:
                    best_key = key
                    best = {
                        'long_threshold': lt,
                        'short_threshold': st,
                        'Validation Precision': diag['Precision'],
                        'Validation Recall': diag['Recall'],
                        'Validation Sharpe': diag['Sharpe'],
                        'Validation Avg Return': diag['Avg Return'],
                        'Validation Trades': diag['Trades'],
                        'Validation Signal Ratio': diag['Signal Ratio'],
                        'Threshold Feasible': True,
                    }
        return best if best is not None else fallback

    raise ValueError(f'Unsupported mode: {mode}')


In [140]:
# -- Bootstrap helpers ----------------------------------
def paper_block_length(n):
    return max(1, int(np.floor(n ** (1 / 3))))


def block_bootstrap_mean_diff(a, b, block=None, n_boot=10000, seed=42, alternative='two-sided'):
    a = np.asarray(pd.Series(a).dropna(), dtype=float)
    b = np.asarray(pd.Series(b).dropna(), dtype=float)
    n = min(len(a), len(b))
    if n < 4:
        return {'mean_diff': np.nan, 'p_value': np.nan, 'block_length': np.nan, 'n_boot': n_boot}
    block = paper_block_length(n) if block is None else int(block)
    n_blocks = n // block
    if n_blocks < 2:
        return {'mean_diff': np.nan, 'p_value': np.nan, 'block_length': block, 'n_boot': n_boot}

    n_trim = n_blocks * block
    a_blocks = a[:n_trim].reshape(n_blocks, block)
    b_blocks = b[:n_trim].reshape(n_blocks, block)
    obs = float(a_blocks.reshape(-1).mean() - b_blocks.reshape(-1).mean())
    rng = np.random.default_rng(seed)
    boot = np.empty(n_boot, dtype=float)
    for i in range(n_boot):
        idx = rng.integers(0, n_blocks, size=n_blocks)
        boot[i] = float(a_blocks[idx].reshape(-1).mean() - b_blocks[idx].reshape(-1).mean())

    if alternative == 'greater':
        p_value = float(np.mean(boot <= 0))
    elif alternative == 'less':
        p_value = float(np.mean(boot >= 0))
    else:
        p_value = float(np.mean(np.abs(boot) >= abs(obs)))
    return {'mean_diff': obs, 'p_value': p_value, 'block_length': block, 'n_boot': n_boot}


def block_bootstrap_mean_vs_zero(a, block=None, n_boot=10000, seed=42, alternative='greater'):
    a = np.asarray(pd.Series(a).dropna(), dtype=float)
    n = len(a)
    if n < 4:
        return {'mean': np.nan, 'p_value': np.nan, 'block_length': np.nan, 'n_boot': n_boot}
    block = paper_block_length(n) if block is None else int(block)
    n_blocks = n // block
    if n_blocks < 2:
        return {'mean': np.nan, 'p_value': np.nan, 'block_length': block, 'n_boot': n_boot}

    n_trim = n_blocks * block
    a_blocks = a[:n_trim].reshape(n_blocks, block)
    obs = float(a_blocks.reshape(-1).mean())
    rng = np.random.default_rng(seed)
    boot = np.empty(n_boot, dtype=float)
    for i in range(n_boot):
        idx = rng.integers(0, n_blocks, size=n_blocks)
        boot[i] = float(a_blocks[idx].reshape(-1).mean())

    if alternative == 'greater':
        p_value = float(np.mean(boot <= 0))
    elif alternative == 'less':
        p_value = float(np.mean(boot >= 0))
    else:
        p_value = float(np.mean(np.abs(boot) >= abs(obs)))
    return {'mean': obs, 'p_value': p_value, 'block_length': block, 'n_boot': n_boot}


## 5. Buy-and-Hold Baseline
Buy-and-hold is exported separately for every asset because it serves both as:
- the passive benchmark in the plots,
- the paired benchmark in the usefulness tests.


In [141]:
def backtest_buy_and_hold(test_df, return_col=RETURN_COL, transaction_cost=TRANSACTION_COST):
    df = test_df.dropna(subset=[return_col]).copy().reset_index(drop=True)
    df['position'] = 1.0
    df['raw_return'] = df[return_col].astype(float)
    if LOG_RETURN_COL in df.columns:
        df['raw_log_return'] = df[LOG_RETURN_COL].astype(float)
    df['position_change'] = (df['position'] - df['position'].shift(1).fillna(0)).abs()
    df['transaction_cost'] = transaction_cost * df['position_change']
    df['gross_return'] = df['position'] * df['raw_return']
    df['net_return'] = df['gross_return'] - df['transaction_cost']
    df['gross_equity_curve'] = (1 + df['gross_return']).cumprod()
    df['net_equity_curve'] = (1 + df['net_return']).cumprod()
    return df


In [142]:
bh_summaries = []

for asset in ASSETS:
    bt = backtest_buy_and_hold(test_data[asset])
    perf = summarize_performance(bt)
    dirm = compute_directional_metrics(bt)
    row = pd.concat([perf, dirm]).to_frame().T
    row.insert(0, 'Asset', asset)
    row.insert(1, 'Strategy', 'Buy-and-Hold')
    bh_summaries.append(row)
    bt.to_csv(RESULT_DIR / 'buy_and_hold' / f'{asset.lower()}_backtest.csv', index=False)

bh_summary = pd.concat(bh_summaries, ignore_index=True)
bh_summary.to_csv(RESULT_DIR / 'buy_and_hold' / 'summary_all_assets.csv', index=False)
print('Buy-and-Hold results:')
display(bh_summary[['Asset', 'Cumulative Return', 'Annualized Sharpe', 'Maximum Drawdown', 'Number of Trades']])


Buy-and-Hold results:


,Asset,Cumulative Return,Annualized Sharpe,Maximum Drawdown,Number of Trades
0,BTC,0.418913,0.841307,-0.347615,1.0
1,ETH,0.199657,0.561407,-0.652824,1.0
2,XRP,2.060773,1.416065,-0.512126,1.0
3,SOL,-0.156708,0.269584,-0.660623,1.0


## 6. Neural Sequence Helpers Used by the Main Grid
Only the shared sequence-building utilities remain here.

The standalone model-specific benchmark branches were removed, so the main experiment grid is the only modeling path that now needs maintenance.


In [143]:
def make_sequence_dataset(df, features, target_col=TARGET_LONG_COL, return_col=RETURN_COL, lookback=24):
    d = df.dropna(subset=features + [target_col, return_col]).reset_index(drop=True).copy()
    X_raw = d[features].values
    y_raw = d[target_col].values.astype(int)
    r_raw = d[return_col].values.astype(float)
    X_seq, y_seq, r_seq = [], [], []
    for i in range(lookback - 1, len(d)):
        X_seq.append(X_raw[i - lookback + 1:i + 1])
        y_seq.append(y_raw[i])
        r_seq.append(r_raw[i])
    return np.array(X_seq), np.array(y_seq), np.array(r_seq), d.iloc[lookback - 1:].reset_index(drop=True)


In [144]:
def build_gru_model(n_steps, n_features):
    model = keras.Sequential([
        layers.Input(shape=(n_steps, n_features)),
        layers.GRU(8, dropout=0.5, recurrent_dropout=0.0),
        layers.Dense(64, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(64, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(1, activation='sigmoid'),
    ])
    model.compile(optimizer='adam', loss='binary_crossentropy')
    return model


In [145]:
def build_sequence_model(arch, n_steps, n_features):
    arch = arch.upper()
    if arch == 'MLP':
        model = keras.Sequential([
            layers.Input(shape=(n_steps, n_features)),
            layers.Flatten(),
            layers.Dense(64, activation='relu'),
            layers.Dropout(0.5),
            layers.Dense(64, activation='relu'),
            layers.Dropout(0.5),
            layers.Dense(1, activation='sigmoid'),
        ])
    elif arch == 'LSTM':
        model = keras.Sequential([
            layers.Input(shape=(n_steps, n_features)),
            layers.LSTM(8, dropout=0.5),
            layers.Dense(64, activation='relu'),
            layers.Dropout(0.5),
            layers.Dense(64, activation='relu'),
            layers.Dropout(0.5),
            layers.Dense(1, activation='sigmoid'),
        ])
    elif arch == 'CNN':
        model = keras.Sequential([
            layers.Input(shape=(n_steps, n_features)),
            layers.Conv1D(8, kernel_size=3, activation='relu', padding='same'),
            layers.MaxPooling1D(pool_size=2, padding='same'),
            layers.Dropout(0.5),
            layers.Conv1D(8, kernel_size=3, activation='relu', padding='same'),
            layers.MaxPooling1D(pool_size=2, padding='same'),
            layers.Dropout(0.5),
            layers.Flatten(),
            layers.Dense(64, activation='relu'),
            layers.Dropout(0.5),
            layers.Dense(64, activation='relu'),
            layers.Dropout(0.5),
            layers.Dense(1, activation='sigmoid'),
        ])
    else:
        raise ValueError(f'Unknown architecture: {arch}')

    model.compile(optimizer='adam', loss='binary_crossentropy')
    return model


## 7. Main Experiment Grid and Standardized Trading Logs
This is the core experimental section of the notebook.

For each asset, feature set, model family, strategy mode, and cost regime, the pipeline:
1. fits separate long and short classifiers for ML models,
2. tunes validation thresholds by precision with guardrails,
3. builds `{0,1}`, `{0,-1}`, or `{0,1,-1}` positions depending on the strategy mode,
4. runs the test backtest,
5. writes one standardized trading log per configuration,
6. stores one summary row for downstream tables and tests.


In [146]:
# -- Main-grid metadata, debug helpers, and caches ------
if RUN_UNIFIED_GRID:
    TABULAR_BUNDLE_CACHE = {}
    RULE_BASE_FRAME_CACHE = {}
    RULE_SIGNAL_CACHE = {}


    def _feature_metadata(feature_set_name):
        return {
            'Input Type': 'candle' if feature_set_name.startswith('candle') else 'ohlc',
            'Feature Type': 'extended' if feature_set_name.endswith('extended') else 'raw',
        }


    def _get_usable_features(asset, feature_set_name):
        feats = FEATURE_SET_DEFS[feature_set_name]
        tr = train_data[asset]
        vl = val_data[asset]
        te = test_data[asset]
        return [c for c in feats if c in tr.columns and c in vl.columns and c in te.columns]


    def _prepare_tabular_bundle(asset, feature_set_name):
        key = (asset, feature_set_name)
        if key in TABULAR_BUNDLE_CACHE:
            return TABULAR_BUNDLE_CACHE[key]

        features = _get_usable_features(asset, feature_set_name)
        if len(features) == 0:
            TABULAR_BUNDLE_CACHE[key] = None
            return None

        required = features + [TARGET_LONG_COL, TARGET_SHORT_COL, RETURN_COL, LOG_RETURN_COL]
        tr = train_data[asset].dropna(subset=required).copy().reset_index(drop=True)
        vl = val_data[asset].dropna(subset=required).copy().reset_index(drop=True)
        te = test_data[asset].dropna(subset=required).copy().reset_index(drop=True)
        if len(tr) == 0 or len(vl) == 0 or len(te) == 0:
            TABULAR_BUNDLE_CACHE[key] = None
            return None

        bundle = {
            'asset': asset,
            'feature_set': feature_set_name,
            'features': features,
            'tr': tr,
            'vl': vl,
            'te': te,
            'X_tr': tr[features],
            'X_vl': vl[features],
            'X_te': te[features],
            'y_tr_long': tr[TARGET_LONG_COL].astype(int).values,
            'y_tr_short': tr[TARGET_SHORT_COL].astype(int).values,
            'vl_ref': vl.reset_index(drop=True),
            'te_ref': te.reset_index(drop=True),
        }
        TABULAR_BUNDLE_CACHE[key] = bundle
        return bundle


    def _get_rule_base_frame(asset, split_name):
        key = (asset, split_name)
        if key in RULE_BASE_FRAME_CACHE:
            return RULE_BASE_FRAME_CACHE[key]
        source = val_data if split_name == 'val' else test_data
        frame = source[asset].dropna(subset=['close', RETURN_COL, LOG_RETURN_COL]).copy().reset_index(drop=True)
        RULE_BASE_FRAME_CACHE[key] = frame
        return frame


In [147]:
# -- Main-grid tabular model helpers --------------------
if RUN_UNIFIED_GRID:
    def _fit_logistic_pair(bundle):
        scaler = StandardScaler()
        X_tr_scaled = scaler.fit_transform(bundle['X_tr'])
        X_vl_scaled = scaler.transform(bundle['X_vl'])
        X_te_scaled = scaler.transform(bundle['X_te'])

        debug_print(f"Fitting Logistic Regression long classifier | rows={len(bundle['X_tr'])} | features={bundle['X_tr'].shape[1]}")
        long_model = LogisticRegression(max_iter=2000, random_state=42, C=1.0, verbose=LOGISTIC_MODEL_VERBOSE)
        debug_print(f"Fitting Logistic Regression short classifier | rows={len(bundle['X_tr'])} | features={bundle['X_tr'].shape[1]}")
        short_model = LogisticRegression(max_iter=2000, random_state=42, C=1.0, verbose=LOGISTIC_MODEL_VERBOSE)
        long_model.fit(X_tr_scaled, bundle['y_tr_long'])
        short_model.fit(X_tr_scaled, bundle['y_tr_short'])

        return {
            'long_prob_v': long_model.predict_proba(X_vl_scaled)[:, 1],
            'short_prob_v': short_model.predict_proba(X_vl_scaled)[:, 1],
            'long_prob_t': long_model.predict_proba(X_te_scaled)[:, 1],
            'short_prob_t': short_model.predict_proba(X_te_scaled)[:, 1],
            'model_params': {},
        }


    def _fit_tree_pair(model_name, bundle):
        if model_name == 'XGBoost':
            debug_print(f"Fitting XGBoost long classifier | rows={len(bundle['X_tr'])} | features={bundle['X_tr'].shape[1]}")
            long_model = xgb.XGBClassifier(n_estimators=100, eval_metric='logloss', random_state=42, n_jobs=TREE_MODEL_N_JOBS, verbosity=XGBOOST_MODEL_VERBOSITY)
            debug_print(f"Fitting XGBoost short classifier | rows={len(bundle['X_tr'])} | features={bundle['X_tr'].shape[1]}")
            short_model = xgb.XGBClassifier(n_estimators=100, eval_metric='logloss', random_state=42, n_jobs=TREE_MODEL_N_JOBS, verbosity=XGBOOST_MODEL_VERBOSITY)
        elif model_name == 'LightGBM':
            debug_print(f"Fitting LightGBM long classifier | rows={len(bundle['X_tr'])} | features={bundle['X_tr'].shape[1]}")
            long_model = lgb.LGBMClassifier(n_estimators=100, random_state=42, n_jobs=TREE_MODEL_N_JOBS, verbosity=LIGHTGBM_MODEL_VERBOSITY)
            debug_print(f"Fitting LightGBM short classifier | rows={len(bundle['X_tr'])} | features={bundle['X_tr'].shape[1]}")
            short_model = lgb.LGBMClassifier(n_estimators=100, random_state=42, n_jobs=TREE_MODEL_N_JOBS, verbosity=LIGHTGBM_MODEL_VERBOSITY)
        else:
            raise ValueError(f'Unsupported tree model: {model_name}')

        long_model.fit(bundle['X_tr'], bundle['y_tr_long'])
        short_model.fit(bundle['X_tr'], bundle['y_tr_short'])

        return {
            'long_prob_v': long_model.predict_proba(bundle['X_vl'])[:, 1],
            'short_prob_v': short_model.predict_proba(bundle['X_vl'])[:, 1],
            'long_prob_t': long_model.predict_proba(bundle['X_te'])[:, 1],
            'short_prob_t': short_model.predict_proba(bundle['X_te'])[:, 1],
            'model_params': {},
        }


    def _run_single_tabular_ml_task(asset, feature_set, model_name):
        debug_print(f'Tabular task start | asset={asset} | feature_set={feature_set} | model={model_name}')
        bundle = _prepare_tabular_bundle(asset, feature_set)
        if bundle is None:
            debug_print(f'Tabular task skipped | asset={asset} | feature_set={feature_set} | model={model_name}')
            return []

        if model_name == 'Logistic Regression':
            probs = _fit_logistic_pair(bundle)
        else:
            probs = _fit_tree_pair(model_name, bundle)

        rows = _evaluate_dual_probabilities(
            asset,
            model_name,
            feature_set,
            probs['long_prob_v'],
            probs['short_prob_v'],
            bundle['vl_ref'],
            probs['long_prob_t'],
            probs['short_prob_t'],
            bundle['te_ref'],
            model_family='ML',
            model_params=probs['model_params'],
        )
        debug_print(f'Tabular task done | asset={asset} | feature_set={feature_set} | model={model_name} | rows={len(rows)}')
        return rows


In [148]:
# -- Main-grid sequence-model helpers -------------------
if RUN_UNIFIED_GRID:
    def _fit_sequence_classifier(model_name, asset, features, target_col, lookback=LOOKBACK_HOURS):
        tr = train_data[asset].dropna(subset=features + [target_col]).copy()
        vl = val_data[asset].dropna(subset=features + [target_col, RETURN_COL, LOG_RETURN_COL]).copy()
        if len(tr) == 0 or len(vl) == 0:
            return None

        scaler = StandardScaler()
        tr_scaled = tr.copy()
        vl_scaled = vl.copy()
        tr_scaled[features] = scaler.fit_transform(tr[features])
        vl_scaled[features] = scaler.transform(vl[features])

        X_tr, y_tr, _, _ = make_sequence_dataset(tr_scaled, features, target_col=target_col, return_col=RETURN_COL, lookback=lookback)
        X_vl, y_vl, _, vl_ref = make_sequence_dataset(vl_scaled, features, target_col=target_col, return_col=RETURN_COL, lookback=lookback)
        if len(X_tr) == 0 or len(X_vl) == 0:
            return None

        if model_name == 'GRU':
            mdl = build_gru_model(lookback, len(features))
        else:
            mdl = build_sequence_model(model_name, lookback, len(features))
        debug_print(f'Fitting {model_name} long/short sequence classifier target={target_col} | train_sequences={len(X_tr)} | val_sequences={len(X_vl)} | features={len(features)}')
        es = keras.callbacks.EarlyStopping(monitor='val_loss', patience=20, restore_best_weights=True)
        mdl.fit(X_tr, y_tr, validation_data=(X_vl, y_vl), epochs=500, batch_size=1024, verbose=SEQUENCE_MODEL_FIT_VERBOSE, callbacks=[es])
        y_prob_v = mdl.predict(X_vl, verbose=0).ravel()
        n = min(len(y_prob_v), len(vl_ref))
        return {
            'model': mdl,
            'scaler': scaler,
            'lookback': lookback,
            'y_prob_v': y_prob_v[:n],
            'vl_ref': vl_ref.iloc[:n].reset_index(drop=True),
        }


    def _predict_sequence_classifier_test(model_obj, asset, features, target_col):
        te = test_data[asset].dropna(subset=features + [target_col, RETURN_COL, LOG_RETURN_COL]).copy().reset_index(drop=True)
        te_scaled = te.copy()
        te_scaled[features] = model_obj['scaler'].transform(te[features])
        X_te, _, _, te_ref = make_sequence_dataset(te_scaled, features, target_col=target_col, return_col=RETURN_COL, lookback=model_obj['lookback'])
        if len(X_te) == 0:
            return np.array([]), te_ref.iloc[:0].copy()
        y_prob_t = model_obj['model'].predict(X_te, verbose=0).ravel()
        n = min(len(y_prob_t), len(te_ref))
        return y_prob_t[:n], te_ref.iloc[:n].reset_index(drop=True)


    def _run_single_sequence_ml_task(asset, feature_set, model_name):
        debug_print(f'Sequence task start | asset={asset} | feature_set={feature_set} | model={model_name}')
        features = _get_usable_features(asset, feature_set)
        if len(features) == 0:
            return []

        long_obj = _fit_sequence_classifier(model_name, asset, features, TARGET_LONG_COL)
        short_obj = _fit_sequence_classifier(model_name, asset, features, TARGET_SHORT_COL)
        if long_obj is None or short_obj is None:
            debug_print(f'Sequence task skipped | asset={asset} | feature_set={feature_set} | model={model_name}')
            return []

        n_val = min(len(long_obj['y_prob_v']), len(short_obj['y_prob_v']), len(long_obj['vl_ref']), len(short_obj['vl_ref']))
        long_prob_v = long_obj['y_prob_v'][:n_val]
        short_prob_v = short_obj['y_prob_v'][:n_val]
        vl_ref = long_obj['vl_ref'].iloc[:n_val].reset_index(drop=True)

        long_prob_t, te_ref_long = _predict_sequence_classifier_test(long_obj, asset, features, TARGET_LONG_COL)
        short_prob_t, te_ref_short = _predict_sequence_classifier_test(short_obj, asset, features, TARGET_SHORT_COL)
        n_test = min(len(long_prob_t), len(short_prob_t), len(te_ref_long), len(te_ref_short))
        long_prob_t = long_prob_t[:n_test]
        short_prob_t = short_prob_t[:n_test]
        te_ref = te_ref_long.iloc[:n_test].reset_index(drop=True)

        rows = _evaluate_dual_probabilities(
            asset,
            model_name,
            feature_set,
            long_prob_v,
            short_prob_v,
            vl_ref,
            long_prob_t,
            short_prob_t,
            te_ref,
            model_family='ML',
            model_params={'lookback_hours': LOOKBACK_HOURS},
        )
        debug_print(f'Sequence task done | asset={asset} | feature_set={feature_set} | model={model_name} | rows={len(rows)}')
        return rows


In [149]:
# -- Main-grid logging and evaluation helpers ----------
if RUN_UNIFIED_GRID:
    def _standardize_trading_log(bt, asset, model_name, feature_set, strategy_mode, cost_label, long_prob, short_prob, thresholds, model_params=None):
        df = bt.copy().reset_index(drop=True)
        if 'datetime' not in df.columns:
            df['datetime'] = pd.NaT
        prev_pos = df['position'].shift(1).fillna(0)

        def _action(p, pp):
            p, pp = int(p), int(pp)
            if p == pp:
                return 'HOLD'
            if pp == 0 and p == 1:
                return 'OPEN_LONG'
            if pp == 0 and p == -1:
                return 'OPEN_SHORT'
            if pp == 1 and p == 0:
                return 'CLOSE_LONG'
            if pp == -1 and p == 0:
                return 'CLOSE_SHORT'
            if pp == 1 and p == -1:
                return 'FLIP_LONG_TO_SHORT'
            if pp == -1 and p == 1:
                return 'FLIP_SHORT_TO_LONG'
            return 'CHANGE'

        df['trade_action'] = [_action(p, pp) for p, pp in zip(df['position'].fillna(0), prev_pos)]
        df['signal'] = np.where(df['position'] > 0, 'LONG', np.where(df['position'] < 0, 'SHORT', 'FLAT'))
        df['is_trade_event'] = (df['position_change'] > 0).astype(int)
        df['asset'] = asset
        df['model'] = model_name
        df['feature_set'] = feature_set
        df['strategy_mode'] = strategy_mode
        df['cost_regime'] = cost_label
        df['y_prob_long'] = np.asarray(long_prob, dtype=float)[:len(df)]
        df['y_prob_short'] = np.asarray(short_prob, dtype=float)[:len(df)] if short_prob is not None else np.nan
        df['long_threshold'] = thresholds.get('long_threshold')
        df['short_threshold'] = thresholds.get('short_threshold')
        df['threshold_feasible'] = thresholds.get('Threshold Feasible', False)
        df['model_params'] = json.dumps(model_params or {}, sort_keys=True)

        out_cols = [
            'datetime', 'asset', 'model', 'feature_set', 'strategy_mode', 'cost_regime',
            'signal', 'trade_action', 'is_trade_event',
            'y_prob_long', 'y_prob_short', 'long_threshold', 'short_threshold', 'threshold_feasible', 'model_params',
            'position', 'position_change', 'transaction_cost',
            'raw_return', 'raw_log_return', 'gross_return', 'net_return', 'gross_equity_curve', 'net_equity_curve',
        ]
        return df[[c for c in out_cols if c in df.columns]].copy()


    def _build_result_row(asset, model_name, feature_set, strategy_mode, cost_label, thresholds, validation_metrics, test_metrics, log_path, model_family='ML', model_params=None):
        meta = _feature_metadata(feature_set)
        return {
            'Asset': asset,
            'Model': model_name,
            'Model Family': model_family,
            'Feature Set': feature_set,
            'Input Type': meta['Input Type'],
            'Feature Type': meta['Feature Type'],
            'Strategy Mode': strategy_mode,
            'Strategy Label': {'long_only': 'long', 'short_only': 'short', 'long_short': 'long_short'}[strategy_mode],
            'Cost Regime': cost_label,
            'Long Threshold': thresholds.get('long_threshold'),
            'Short Threshold': thresholds.get('short_threshold'),
            'Threshold Feasible': thresholds.get('Threshold Feasible', False),
            'Validation Sharpe': validation_metrics['Sharpe'],
            'Validation Precision': validation_metrics['Precision'],
            'Validation Recall': validation_metrics['Recall'],
            'Validation Avg Return': validation_metrics['Avg Return'],
            'Validation Trades': validation_metrics['Trades'],
            'Validation Signal Ratio': validation_metrics['Signal Ratio'],
            'Test Sharpe': test_metrics['Sharpe'],
            'Test Precision': test_metrics['Precision'],
            'Test Recall': test_metrics['Recall'],
            'Test Avg Return': test_metrics['Avg Return'],
            'Cumulative Return': test_metrics['Cumulative Return'],
            'Maximum Drawdown': test_metrics['Maximum Drawdown'],
            'Turnover': test_metrics['Turnover'],
            'Number of Trades': test_metrics['Number of Trades'],
            'Annualized Sharpe': test_metrics['Annualized Sharpe'],
            'Model Params': json.dumps(model_params or {}, sort_keys=True),
            'Log Path': str(log_path),
        }


    def _evaluate_dual_probabilities(asset, model_name, feature_set, long_prob_v, short_prob_v, vl_ref, long_prob_t, short_prob_t, te_ref, model_family='ML', model_params=None):
        rows = []
        for strategy_mode in UNIFIED_STRATEGY_MODES:
            for cost_label, cost_value in COST_REGIMES.items():
                thresholds = tune_threshold_precision(
                    long_prob_v,
                    short_prob_v,
                    vl_ref,
                    mode=strategy_mode,
                    long_grid=LONG_T_GRID,
                    short_grid=SHORT_T_GRID,
                    cost=cost_value,
                )
                pos_t = build_positions_from_dual_probabilities(
                    long_prob_t,
                    short_prob_t,
                    mode=strategy_mode,
                    long_threshold=thresholds.get('long_threshold'),
                    short_threshold=thresholds.get('short_threshold'),
                )
                bt_t = run_backtest_from_position(te_ref, pos_t, return_col=RETURN_COL, cost=cost_value)
                validation_metrics = {
                    'Sharpe': thresholds.get('Validation Sharpe'),
                    'Precision': thresholds.get('Validation Precision'),
                    'Recall': thresholds.get('Validation Recall'),
                    'Avg Return': thresholds.get('Validation Avg Return'),
                    'Trades': thresholds.get('Validation Trades'),
                    'Signal Ratio': thresholds.get('Validation Signal Ratio'),
                }
                test_metrics = summarize_signal_performance(bt_t, mode=strategy_mode)
                out_dir = DATA_LOG_DIR / cost_label / strategy_mode / model_name.lower().replace(' ', '_') / feature_set / asset.lower()
                out_dir.mkdir(parents=True, exist_ok=True)
                out_path = out_dir / 'trading_log.csv'
                log_df = _standardize_trading_log(
                    bt_t,
                    asset,
                    model_name,
                    feature_set,
                    strategy_mode,
                    cost_label,
                    long_prob_t,
                    short_prob_t,
                    thresholds,
                    model_params=model_params,
                )
                log_df.to_csv(out_path, index=False)
                rows.append(
                    _build_result_row(
                        asset,
                        model_name,
                        feature_set,
                        strategy_mode,
                        cost_label,
                        thresholds,
                        validation_metrics,
                        test_metrics,
                        out_path,
                        model_family=model_family,
                        model_params=model_params,
                    )
                )
        return rows


In [150]:
# -- Main-grid rule helpers and caching -----------------
if RUN_UNIFIED_GRID:
    def _rule_signal_dataframe(df, model_name, params):
        work = df[['datetime', RETURN_COL, LOG_RETURN_COL]].copy().reset_index(drop=True)
        price = df['close'].astype(float).reset_index(drop=True)

        if model_name == 'SMA':
            sma_short = price.rolling(params['short_window'], min_periods=params['short_window']).mean()
            sma_long = price.rolling(params['long_window'], min_periods=params['long_window']).mean()
            score = sma_short / (sma_long + EPS) - 1
        elif model_name == 'RSI':
            rsi = compute_rsi(price, params['window'])
            score = (50 - rsi) / 50
        else:
            raise ValueError(f'Unsupported rule model: {model_name}')

        work['long_prob'] = _sigmoid_score(score.fillna(0.0), scale=20.0)
        work['short_prob'] = _sigmoid_score((-score).fillna(0.0), scale=20.0)
        return work.loc[score.notna()].reset_index(drop=True)


    def _rule_param_grid(model_name):
        if model_name == 'SMA':
            return [
                {'short_window': sw, 'long_window': lw}
                for sw in SMA_SHORT_WINDOWS for lw in SMA_LONG_WINDOWS if sw < lw
            ]
        if model_name == 'RSI':
            return [{'window': w} for w in RSI_WINDOWS]
        raise ValueError(f'Unsupported rule model: {model_name}')


    def _rule_signal_key(asset, split_name, model_name, params):
        return (asset, split_name, model_name, tuple(sorted(params.items())))


    def _get_rule_signal_frame(asset, split_name, model_name, params):
        key = _rule_signal_key(asset, split_name, model_name, params)
        if key in RULE_SIGNAL_CACHE:
            return RULE_SIGNAL_CACHE[key]
        signal_df = _rule_signal_dataframe(_get_rule_base_frame(asset, split_name), model_name, params)
        RULE_SIGNAL_CACHE[key] = signal_df
        return signal_df


    def _warm_rule_signal_cache():
        debug_print('Warming rule-signal cache...')
        for asset in ASSETS:
            _get_rule_base_frame(asset, 'val')
            _get_rule_base_frame(asset, 'test')
            for model_name in RULE_MODELS:
                for params in _rule_param_grid(model_name):
                    _get_rule_signal_frame(asset, 'val', model_name, params)
                    _get_rule_signal_frame(asset, 'test', model_name, params)


    def _run_single_rule_task(asset, model_name, strategy_mode, cost_label):
        debug_print(f'Rule task start | asset={asset} | model={model_name} | mode={strategy_mode} | cost={cost_label}')
        cost_value = COST_REGIMES[cost_label]
        best_key = None
        best_payload = None

        for params in _rule_param_grid(model_name):
            vl_sig = _get_rule_signal_frame(asset, 'val', model_name, params)
            te_sig = _get_rule_signal_frame(asset, 'test', model_name, params)
            if len(vl_sig) == 0 or len(te_sig) == 0:
                continue

            thresholds = tune_threshold_precision(
                vl_sig['long_prob'].values,
                vl_sig['short_prob'].values,
                vl_sig[['datetime', RETURN_COL, LOG_RETURN_COL]].copy(),
                mode=strategy_mode,
                cost=cost_value,
            )
            if not thresholds.get('Threshold Feasible', False):
                continue

            key = (
                thresholds.get('Validation Precision') if pd.notna(thresholds.get('Validation Precision')) else -np.inf,
                thresholds.get('Validation Recall') if pd.notna(thresholds.get('Validation Recall')) else -np.inf,
                thresholds.get('Validation Sharpe') if pd.notna(thresholds.get('Validation Sharpe')) else -np.inf,
            )
            if best_key is None or key > best_key:
                best_key = key
                best_payload = (params, thresholds, vl_sig, te_sig)

        if best_payload is None:
            debug_print(f'Rule task skipped | asset={asset} | model={model_name} | mode={strategy_mode} | cost={cost_label}')
            return []

        params, thresholds, _, te_sig = best_payload
        pos_t = build_positions_from_dual_probabilities(
            te_sig['long_prob'].values,
            te_sig['short_prob'].values,
            mode=strategy_mode,
            long_threshold=thresholds.get('long_threshold'),
            short_threshold=thresholds.get('short_threshold'),
        )
        te_ref = te_sig[['datetime', RETURN_COL, LOG_RETURN_COL]].copy().reset_index(drop=True)
        bt_t = run_backtest_from_position(te_ref, pos_t, return_col=RETURN_COL, cost=cost_value)
        validation_metrics = {
            'Sharpe': thresholds.get('Validation Sharpe'),
            'Precision': thresholds.get('Validation Precision'),
            'Recall': thresholds.get('Validation Recall'),
            'Avg Return': thresholds.get('Validation Avg Return'),
            'Trades': thresholds.get('Validation Trades'),
            'Signal Ratio': thresholds.get('Validation Signal Ratio'),
        }
        test_metrics = summarize_signal_performance(bt_t, mode=strategy_mode)

        rows = []
        for feature_set in UNIFIED_FEATURE_SETS:
            out_dir = DATA_LOG_DIR / cost_label / strategy_mode / model_name.lower() / feature_set / asset.lower()
            out_dir.mkdir(parents=True, exist_ok=True)
            out_path = out_dir / 'trading_log.csv'
            log_df = _standardize_trading_log(
                bt_t,
                asset,
                model_name,
                feature_set,
                strategy_mode,
                cost_label,
                te_sig['long_prob'].values,
                te_sig['short_prob'].values,
                thresholds,
                model_params=params,
            )
            log_df.to_csv(out_path, index=False)
            rows.append(
                _build_result_row(
                    asset,
                    model_name,
                    feature_set,
                    strategy_mode,
                    cost_label,
                    thresholds,
                    validation_metrics,
                    test_metrics,
                    out_path,
                    model_family='Rule',
                    model_params=params,
                )
            )
        debug_print(f'Rule task done | asset={asset} | model={model_name} | mode={strategy_mode} | cost={cost_label} | rows={len(rows)}')
        return rows


In [ ]:
# GPU support

print("TF version:", tf.__version__)
print("Built with CUDA:", tf.test.is_built_with_cuda())
print("Visible GPUs:", tf.config.list_physical_devices('GPU'))
print("GPU device name:", tf.test.gpu_device_name())

# show where ops are actually placed
tf.debugging.set_log_device_placement(True)
a = tf.constant([[1.0, 2.0], [3.0, 4.0]])
b = tf.constant([[1.0], [1.0]])
print(tf.matmul(a, b))


In [ ]:
# -- Main-grid execution --------------------------------
if RUN_UNIFIED_GRID:
    debug_print('Warming tabular bundle cache...')
    for asset in ASSETS:
        for feature_set in UNIFIED_FEATURE_SETS:
            _prepare_tabular_bundle(asset, feature_set)

    _warm_rule_signal_cache()

    rows = []

    tabular_tasks = [
        (asset, feature_set, model_name)
        for asset in ASSETS
        for feature_set in UNIFIED_FEATURE_SETS
        for model_name in TABULAR_ML_MODELS
    ]
    debug_print(f'Launching {len(tabular_tasks)} tabular ML tasks with {UNIFIED_PARALLEL_JOBS} worker(s)')
    if len(tabular_tasks) > 0:
        tabular_batches = joblib.Parallel(n_jobs=UNIFIED_PARALLEL_JOBS, prefer='threads')(
            joblib.delayed(_run_single_tabular_ml_task)(asset, feature_set, model_name)
            for asset, feature_set, model_name in tabular_tasks
        )
        for batch in tabular_batches:
            rows.extend(batch)

    sequence_tasks = [
        (asset, feature_set, model_name)
        for asset in ASSETS
        for feature_set in UNIFIED_FEATURE_SETS
        for model_name in SEQUENCE_ML_MODELS
    ]
    debug_print(f'Running {len(sequence_tasks)} sequence-model tasks sequentially')
    for asset, feature_set, model_name in sequence_tasks:
        rows.extend(_run_single_sequence_ml_task(asset, feature_set, model_name))

    rule_tasks = [
        (asset, model_name, strategy_mode, cost_label)
        for asset in ASSETS
        for model_name in RULE_MODELS
        for strategy_mode in UNIFIED_STRATEGY_MODES
        for cost_label in COST_REGIMES.keys()
    ]
    debug_print(f'Launching {len(rule_tasks)} rule tasks with {RULE_PARALLEL_JOBS} worker(s)')
    if len(rule_tasks) > 0:
        rule_batches = joblib.Parallel(n_jobs=RULE_PARALLEL_JOBS, prefer='threads')(
            joblib.delayed(_run_single_rule_task)(asset, model_name, strategy_mode, cost_label)
            for asset, model_name, strategy_mode, cost_label in rule_tasks
        )
        for batch in rule_batches:
            rows.extend(batch)

    unified_summary = pd.DataFrame(rows)
    unified_summary.to_csv(DATA_LOG_DIR / 'unified_experiment_summary.csv', index=False)
    print('Main experiment grid completed. Logs saved under data/trading_logs/.')
    display(unified_summary.head(20))
else:
    print('RUN_UNIFIED_GRID=False, skipping main experiment grid.')


[DEBUG] Warming tabular bundle cache...
[DEBUG] Warming rule-signal cache...
[DEBUG] Launching 48 tabular ML tasks with 4 worker(s)
[DEBUG] Tabular task start | asset=BTC | feature_set=ohlc_raw | model=LightGBM
[DEBUG] Fitting LightGBM long classifier | rows=21868 | features=8
[DEBUG] Tabular task start | asset=BTC | feature_set=candle_raw | model=Logistic Regression
[DEBUG] Tabular task start | asset=BTC | feature_set=ohlc_raw | model=Logistic Regression
[DEBUG] Fitting LightGBM short classifier | rows=21868 | features=8
[DEBUG] Tabular task start | asset=BTC | feature_set=ohlc_raw | model=XGBoost
[DEBUG] Fitting XGBoost long classifier | rows=21868 | features=8
[DEBUG] Fitting XGBoost short classifier | rows=21868 | features=8
[DEBUG] Fitting Logistic Regression long classifier | rows=21868 | features=8
[DEBUG] Fitting Logistic Regression short classifier | rows=21868 | features=8
[DEBUG] Fitting Logistic Regression long classifier | rows=21868 | features=8
[DEBUG] Fitting Logistic R

[03:51:32] INFO: /Users/runner/work/xgboost/xgboost/src/data/iterative_dmatrix.cc:56: Finished constructing the `IterativeDMatrix`: (21868, 8, 174944).


[LightGBM] [Info] Number of positive: 10858, number of negative: 11010
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000912 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1571
[LightGBM] [Info] Number of data points in the train set: 21868, number of used features: 8
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.496525 -> initscore=-0.013902
[LightGBM] [Info] Start training from score -0.013902


[03:51:32] INFO: /Users/runner/work/xgboost/xgboost/src/data/iterative_dmatrix.cc:56: Finished constructing the `IterativeDMatrix`: (21868, 8, 174944).


[DEBUG] Tabular task done | asset=BTC | feature_set=ohlc_raw | model=Logistic Regression | rows=6
[DEBUG] Tabular task start | asset=BTC | feature_set=candle_raw | model=XGBoost
[DEBUG] Fitting XGBoost long classifier | rows=21868 | features=8
[DEBUG] Fitting XGBoost short classifier | rows=21868 | features=8


...

[DEBUG] Tabular task done | asset=SOL | feature_set=ohlc_raw | model=XGBoost | rows=6
[DEBUG] Tabular task start | asset=SOL | feature_set=candle_raw | model=LightGBM
[DEBUG] Fitting LightGBM long classifier | rows=21868 | features=8
[DEBUG] Fitting LightGBM short classifier | rows=21868 | features=8
[LightGBM] [Info] Number of positive: 10755, number of negative: 11113
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000160 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1571
[LightGBM] [Info] Number of data points in the train set: 21868, number of used features: 8
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.491815 -> initscore=-0.032745
[LightGBM] [Info] Start training from score -0.032745
[LightGBM] [Info] Number of positive: 10819, number of negative: 11049
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead

[03:56:57] INFO: /Users/runner/work/xgboost/xgboost/src/data/iterative_dmatrix.cc:56: Finished constructing the `IterativeDMatrix`: (21868, 21, 459228).
[03:56:57] INFO: /Users/runner/work/xgboost/xgboost/src/data/iterative_dmatrix.cc:56: Finished constructing the `IterativeDMatrix`: (21868, 21, 459228).


[DEBUG] Tabular task done | asset=SOL | feature_set=candle_raw | model=XGBoost | rows=6
[DEBUG] Tabular task start | asset=SOL | feature_set=ohlc_extended | model=LightGBM
[DEBUG] Fitting LightGBM long classifier | rows=21868 | features=21
[DEBUG] Fitting LightGBM short classifier | rows=21868 | features=21
[LightGBM] [Info] Number of positive: 10755, number of negative: 11113
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000402 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 4886
[LightGBM] [Info] Number of data points in the train set: 21868, number of used features: 21
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.491815 -> initscore=-0.032745
[LightGBM] [Info] Start training from score -0.032745
[LightGBM] [Info] Number of positive: 10819, number of negative: 11049
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the 

[03:57:25] INFO: /Users/runner/work/xgboost/xgboost/src/data/iterative_dmatrix.cc:56: Finished constructing the `IterativeDMatrix`: (21868, 21, 459228).
[03:57:25] INFO: /Users/runner/work/xgboost/xgboost/src/data/iterative_dmatrix.cc:56: Finished constructing the `IterativeDMatrix`: (21868, 21, 459228).


[DEBUG] Tabular task done | asset=SOL | feature_set=ohlc_extended | model=XGBoost | rows=6
[DEBUG] Tabular task start | asset=SOL | feature_set=candle_extended | model=LightGBM
[DEBUG] Fitting LightGBM long classifier | rows=21868 | features=21
[DEBUG] Fitting LightGBM short classifier | rows=21868 | features=21
[LightGBM] [Info] Number of positive: 10755, number of negative: 11113
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.003278 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 4886
[LightGBM] [Info] Number of data points in the train set: 21868, number of used features: 21
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.491815 -> initscore=-0.032745
[LightGBM] [Info] Start training from score -0.032745
[LightGBM] [Info] Number of positive: 10819, number of negative: 11049
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000474 seconds.
You can set `forc

## 8. Tables and Aggregated Comparisons
The reporting layer converts the full configuration grid into tables that are easier to compare with the FRL paper.

Outputs include:
- detailed long-only, short-only, and long-short configuration tables,
- grouped `candle vs ohlc` and `raw vs extended` summaries,
- average performance by model family,
- per-asset exports and all-assets exports,
- direct `with_cost` vs `no_cost` comparison tables.


In [ ]:
# -- Table helper definitions ---------------------------
if 'unified_summary' not in globals() or unified_summary is None or len(unified_summary) == 0:
    p = DATA_LOG_DIR / 'unified_experiment_summary.csv'
    unified_summary = pd.read_csv(p) if p.exists() else pd.DataFrame()


def _sort_detail_table(df):
    return df.sort_values(['Test Precision', 'Validation Precision', 'Test Sharpe'], ascending=[False, False, False]).reset_index(drop=True)


def _paper_metric_summary(df, group_cols):
    metric_cols = [
        'Validation Sharpe', 'Validation Precision', 'Validation Recall', 'Validation Avg Return',
        'Test Sharpe', 'Test Precision', 'Test Recall', 'Test Avg Return',
    ]
    return (df.groupby(group_cols, as_index=False)[metric_cols]
              .mean()
              .sort_values(group_cols)
              .reset_index(drop=True))


def _select_plot_configs(df):
    ranked = df.copy()
    for col in ['Validation Precision', 'Validation Recall', 'Validation Sharpe', 'Test Sharpe', 'Test Avg Return']:
        ranked[col] = pd.to_numeric(ranked[col], errors='coerce')
    ranked = ranked.sort_values(
        ['Asset', 'Model', 'Strategy Mode', 'Cost Regime',
         'Validation Precision', 'Validation Recall', 'Validation Sharpe', 'Test Sharpe', 'Test Avg Return'],
        ascending=[True, True, True, True, False, False, False, False, False],
        na_position='last',
    )
    return ranked.drop_duplicates(subset=['Asset', 'Model', 'Strategy Mode', 'Cost Regime'], keep='first').reset_index(drop=True)


def _iter_scopes(df):
    yield 'all_assets', df.copy()
    for asset in ASSETS:
        asset_df = df[df['Asset'] == asset].copy()
        if len(asset_df) > 0:
            yield asset.lower(), asset_df


def _strategy_table_tag(strategy_mode):
    return {
        'long_only': 'table8',
        'short_only': 'table9',
        'long_short': 'table_long_short',
    }[strategy_mode]


def _paper_detail_view(df, include_asset=False):
    cols = (['Asset'] if include_asset else []) + [
        'Model', 'Input Type', 'Feature Type', 'Strategy Label',
        'Validation Sharpe', 'Validation Precision', 'Validation Recall', 'Validation Avg Return',
        'Test Sharpe', 'Test Precision', 'Test Recall', 'Test Avg Return',
    ]
    out = df.reindex(columns=cols).rename(columns={'Strategy Label': 'Strategy'}).copy()
    sort_cols = (['Asset'] if include_asset else []) + ['Test Precision', 'Validation Precision', 'Test Sharpe']
    ascending = ([True] if include_asset else []) + [False, False, False]
    return out.sort_values(sort_cols, ascending=ascending).reset_index(drop=True)


In [ ]:
# -- Table execution ------------------------------------
if len(unified_summary) == 0:
    print('No unified results available yet.')
    comparison = pd.DataFrame()
    plot_configs = pd.DataFrame()
else:
    comparison = unified_summary.copy()
    numeric_cols = [
        'Validation Sharpe', 'Validation Precision', 'Validation Recall', 'Validation Avg Return',
        'Validation Trades', 'Validation Signal Ratio', 'Test Sharpe', 'Test Precision',
        'Test Recall', 'Test Avg Return', 'Cumulative Return', 'Maximum Drawdown',
        'Turnover', 'Number of Trades', 'Annualized Sharpe',
    ]
    for col in numeric_cols:
        if col in comparison.columns:
            comparison[col] = pd.to_numeric(comparison[col], errors='coerce')

    if 'Model Family' not in comparison.columns:
        comparison['Model Family'] = np.where(comparison['Model'].isin(RULE_MODELS), 'Rule', 'ML')
    if 'Input Type' not in comparison.columns and 'Feature Set' in comparison.columns:
        comparison['Input Type'] = np.where(comparison['Feature Set'].astype(str).str.startswith('candle'), 'candle', 'ohlc')
    if 'Feature Type' not in comparison.columns and 'Feature Set' in comparison.columns:
        comparison['Feature Type'] = np.where(comparison['Feature Set'].astype(str).str.endswith('extended'), 'extended', 'raw')
    if 'Strategy Label' not in comparison.columns and 'Strategy Mode' in comparison.columns:
        comparison['Strategy Label'] = comparison['Strategy Mode'].map({
            'long_only': 'long',
            'short_only': 'short',
            'long_short': 'long_short',
        })
    if 'Threshold Feasible' not in comparison.columns:
        comparison['Threshold Feasible'] = np.nan
    if 'Annualized Sharpe' not in comparison.columns:
        comparison['Annualized Sharpe'] = np.nan
    if 'Model Params' not in comparison.columns:
        comparison['Model Params'] = '{}'

    comparison = comparison.sort_values(['Asset', 'Model Family', 'Model', 'Feature Set', 'Strategy Mode', 'Cost Regime']).reset_index(drop=True)
    comparison.to_csv(RESULT_DIR / 'summary_all_strategies_raw.csv', index=False)

    detail_cols = [
        'Asset', 'Model', 'Model Family', 'Input Type', 'Feature Type', 'Feature Set', 'Strategy Label',
        'Cost Regime', 'Threshold Feasible',
        'Validation Sharpe', 'Validation Precision', 'Validation Recall', 'Validation Avg Return',
        'Test Sharpe', 'Test Precision', 'Test Recall', 'Test Avg Return', 'Cumulative Return',
        'Maximum Drawdown', 'Turnover', 'Number of Trades', 'Annualized Sharpe', 'Model Params',
    ]
    detail_all = _sort_detail_table(comparison.reindex(columns=detail_cols))
    detail_all.to_csv(RESULT_DIR / 'paper_tables' / 'table_detail_all_configs.csv', index=False)

    grouped_exports = {'table10': [], 'table11': [], 'table12': []}
    preview_tables = []

    for scope_name, scope_df in _iter_scopes(comparison):
        include_asset = scope_name == 'all_assets'
        for strategy_mode in UNIFIED_STRATEGY_MODES:
            for cost_regime in COST_REGIMES.keys():
                sub = scope_df[(scope_df['Strategy Mode'] == strategy_mode) & (scope_df['Cost Regime'] == cost_regime)].copy()
                if len(sub) == 0:
                    continue

                detail_view = _paper_detail_view(sub, include_asset=include_asset)
                detail_name = f'{_strategy_table_tag(strategy_mode)}_{scope_name}_{cost_regime}.csv'
                detail_view.to_csv(RESULT_DIR / 'paper_tables' / detail_name, index=False)

                ml_sub = sub[sub['Model Family'] == 'ML'].copy()
                if len(ml_sub) > 0:
                    table10 = _paper_metric_summary(ml_sub, ['Input Type'])
                    table10.to_csv(RESULT_DIR / 'paper_tables' / f'table10_representation_{scope_name}_{strategy_mode}_{cost_regime}.csv', index=False)
                    grouped_exports['table10'].append(table10.assign(Scope=scope_name, Strategy_Mode=strategy_mode, Cost_Regime=cost_regime))
                    if len(preview_tables) < 4:
                        preview_tables.append((f'Table 10 | {scope_name} | {strategy_mode} | {cost_regime}', table10))

                    table11 = _paper_metric_summary(ml_sub, ['Feature Type'])
                    table11.to_csv(RESULT_DIR / 'paper_tables' / f'table11_feature_type_{scope_name}_{strategy_mode}_{cost_regime}.csv', index=False)
                    grouped_exports['table11'].append(table11.assign(Scope=scope_name, Strategy_Mode=strategy_mode, Cost_Regime=cost_regime))
                    if len(preview_tables) < 6:
                        preview_tables.append((f'Table 11 | {scope_name} | {strategy_mode} | {cost_regime}', table11))

                table12 = _paper_metric_summary(sub, ['Model'])
                table12.to_csv(RESULT_DIR / 'paper_tables' / f'table12_model_average_{scope_name}_{strategy_mode}_{cost_regime}.csv', index=False)
                grouped_exports['table12'].append(table12.assign(Scope=scope_name, Strategy_Mode=strategy_mode, Cost_Regime=cost_regime))

    for name, frames in grouped_exports.items():
        if len(frames) > 0:
            pd.concat(frames, ignore_index=True).to_csv(RESULT_DIR / 'paper_tables' / f'{name}_bundle.csv', index=False)

    cost_compare = comparison.pivot_table(
        index=['Asset', 'Model', 'Model Family', 'Feature Set', 'Input Type', 'Feature Type', 'Strategy Mode'],
        columns='Cost Regime',
        values=['Test Sharpe', 'Test Avg Return', 'Cumulative Return', 'Turnover', 'Number of Trades'],
    )
    cost_compare.columns = [' '.join(col).strip() for col in cost_compare.columns.to_flat_index()]
    cost_compare = cost_compare.reset_index()
    if 'Test Sharpe no_cost' in cost_compare.columns and 'Test Sharpe with_cost' in cost_compare.columns:
        cost_compare['Sharpe Cost Drag'] = cost_compare['Test Sharpe no_cost'] - cost_compare['Test Sharpe with_cost']
    if 'Test Avg Return no_cost' in cost_compare.columns and 'Test Avg Return with_cost' in cost_compare.columns:
        cost_compare['Avg Return Cost Drag'] = cost_compare['Test Avg Return no_cost'] - cost_compare['Test Avg Return with_cost']
    if 'Cumulative Return no_cost' in cost_compare.columns and 'Cumulative Return with_cost' in cost_compare.columns:
        cost_compare['Cumulative Return Cost Drag'] = cost_compare['Cumulative Return no_cost'] - cost_compare['Cumulative Return with_cost']
    cost_compare.to_csv(RESULT_DIR / 'paper_tables' / 'cost_comparison_by_configuration.csv', index=False)

    plot_configs = _select_plot_configs(comparison)
    plot_configs.to_csv(RESULT_DIR / 'paper_tables' / 'plot_configs_best_by_model_asset_strategy_cost.csv', index=False)

    print('===== PAPER-STYLE TABLES SAVED =====')
    display(detail_all.head(20))
    print('===== SAMPLE GROUPED TABLE OUTPUTS =====')
    for title, tbl in preview_tables:
        print(title)
        display(tbl)


## 9. Equity Curves and Bootstrap-Based Statistical Tests
This section focuses on interpretation rather than raw enumeration.

It draws best-configuration equity curves for each asset and then runs two types of statistical checks:
- paper-style grouped comparisons such as `candle vs ohlc` and `extended vs raw`,
- usefulness tests asking whether a selected strategy has positive mean net return and whether it outperforms buy-and-hold.


In [ ]:
# -- Plotting and stat-test helpers ---------------------
if 'comparison' not in globals() or comparison is None or len(comparison) == 0:
    p = DATA_LOG_DIR / 'unified_experiment_summary.csv'
    comparison = pd.read_csv(p) if p.exists() else pd.DataFrame()
if 'plot_configs' not in globals() or plot_configs is None or len(plot_configs) == 0:
    plot_configs = _select_plot_configs(comparison) if len(comparison) > 0 else pd.DataFrame()


def _load_log_returns(log_path):
    lp = Path(log_path)
    if not lp.exists():
        return None
    df = pd.read_csv(lp)
    return df['net_return'].dropna().reset_index(drop=True)


def _group_mean_returns(df):
    series_list = []
    for _, row in df.iterrows():
        s = _load_log_returns(row['Log Path'])
        if s is not None and len(s) > 0:
            series_list.append(s)
    if len(series_list) == 0:
        return pd.Series(dtype=float)
    n = min(len(s) for s in series_list)
    arr = np.column_stack([s.iloc[:n].values for s in series_list])
    return pd.Series(arr.mean(axis=1))


def _scope_subset(df, scope_name):
    if scope_name == 'all_assets':
        return df.copy()
    return df[df['Asset'] == scope_name.upper()].copy()


def _group_bootstrap_task(scope_name, strategy_mode, cost_regime, test_type):
    scope_df = _scope_subset(comparison, scope_name)
    ml_sub = scope_df[
        (scope_df['Model Family'] == 'ML') &
        (scope_df['Strategy Mode'] == strategy_mode) &
        (scope_df['Cost Regime'] == cost_regime)
    ].copy()
    if len(ml_sub) == 0:
        return None

    if test_type == 'candle_vs_ohlc':
        left = _group_mean_returns(ml_sub[ml_sub['Input Type'] == 'candle'])
        right = _group_mean_returns(ml_sub[ml_sub['Input Type'] == 'ohlc'])
    elif test_type == 'extended_vs_raw':
        left = _group_mean_returns(ml_sub[ml_sub['Feature Type'] == 'extended'])
        right = _group_mean_returns(ml_sub[ml_sub['Feature Type'] == 'raw'])
    else:
        raise ValueError(f'Unsupported grouped bootstrap test: {test_type}')

    if len(left) == 0 or len(right) == 0:
        return None

    res = block_bootstrap_mean_diff(left, right)
    return {
        'Scope': scope_name,
        'Test Type': test_type,
        'Strategy Mode': strategy_mode,
        'Cost Regime': cost_regime,
        'Mean Net Return Diff': res['mean_diff'],
        'Bootstrap p-value': res['p_value'],
        'Block Length': res['block_length'],
        'Bootstrap Iterations': res['n_boot'],
    }


def _usefulness_task(row):
    strat_ret = _load_log_returns(row['Log Path'])
    if strat_ret is None or len(strat_ret) == 0:
        return None
    zero_test = block_bootstrap_mean_vs_zero(strat_ret)
    bh_bt = backtest_buy_and_hold(test_data[row['Asset']], transaction_cost=COST_REGIMES[row['Cost Regime']])
    bh_ret = bh_bt['net_return'].dropna().reset_index(drop=True)
    diff_test = block_bootstrap_mean_diff(strat_ret, bh_ret, alternative='greater')
    return {
        'Asset': row['Asset'],
        'Model': row['Model'],
        'Feature Set': row['Feature Set'],
        'Strategy Mode': row['Strategy Mode'],
        'Cost Regime': row['Cost Regime'],
        'Mean Net Return': zero_test['mean'],
        'p-value vs 0': zero_test['p_value'],
        'Mean Excess Return vs BuyHold': diff_test['mean_diff'],
        'p-value vs BuyHold': diff_test['p_value'],
        'Block Length': zero_test['block_length'],
        'Bootstrap Iterations': zero_test['n_boot'],
    }


In [ ]:
# -- Plotting and stat-test execution -------------------
if len(plot_configs) == 0:
    print('No unified summary available for plots/stat tests.')
else:
    for strategy_mode in UNIFIED_STRATEGY_MODES:
        for cost_regime in COST_REGIMES.keys():
            fig, axes = plt.subplots(2, 2, figsize=(14, 10))
            axes = axes.flatten()
            for i, asset in enumerate(ASSETS):
                ax = axes[i]
                sub = plot_configs[
                    (plot_configs['Asset'] == asset) &
                    (plot_configs['Strategy Mode'] == strategy_mode) &
                    (plot_configs['Cost Regime'] == cost_regime)
                ].copy()
                sub = sub.sort_values('Validation Precision', ascending=False)
                for _, row in sub.iterrows():
                    lp = Path(row['Log Path'])
                    if lp.exists():
                        bt = pd.read_csv(lp)
                        ax.plot(bt['net_equity_curve'].values, label=f"{row['Model']} [{row['Feature Set']}]")

                bh_bt = backtest_buy_and_hold(test_data[asset], transaction_cost=COST_REGIMES[cost_regime])
                ax.plot(bh_bt['net_equity_curve'].values, color='black', linestyle='--', linewidth=1.4, label='Buy&Hold')
                ax.set_title(f"{asset} | {strategy_mode} | {cost_regime}")
                ax.set_ylabel('Equity (start=1)')
                ax.set_xlabel('Hour')
                ax.legend(fontsize=7)

            plt.tight_layout()
            out_name = f'equity_curves_{strategy_mode}_{cost_regime}.png'
            plt.savefig(RESULT_DIR / out_name, dpi=120)
            plt.show()
            print(f'Saved {out_name}')

    scope_names = ['all_assets'] + [asset.lower() for asset in ASSETS]
    grouped_tasks = [
        (scope_name, strategy_mode, cost_regime, test_type)
        for scope_name in scope_names
        for strategy_mode in UNIFIED_STRATEGY_MODES
        for cost_regime in COST_REGIMES.keys()
        for test_type in ['candle_vs_ohlc', 'extended_vs_raw']
    ]
    bootstrap_rows = joblib.Parallel(n_jobs=BOOTSTRAP_PARALLEL_JOBS, prefer='threads')(
        joblib.delayed(_group_bootstrap_task)(scope_name, strategy_mode, cost_regime, test_type)
        for scope_name, strategy_mode, cost_regime, test_type in grouped_tasks
    )
    bootstrap_rows = [row for row in bootstrap_rows if row is not None]

    usefulness_rows = joblib.Parallel(n_jobs=BOOTSTRAP_PARALLEL_JOBS, prefer='threads')(
        joblib.delayed(_usefulness_task)(row)
        for row in plot_configs.to_dict('records')
    )
    usefulness_rows = [row for row in usefulness_rows if row is not None]

    bootstrap_df = pd.DataFrame(bootstrap_rows)
    usefulness_df = pd.DataFrame(usefulness_rows)
    bootstrap_df.to_csv(RESULT_DIR / 'stat_tests' / 'paper_group_bootstrap_tests.csv', index=False)
    usefulness_df.to_csv(RESULT_DIR / 'stat_tests' / 'usefulness_tests_best_configs.csv', index=False)
    print('Saved grouped bootstrap tests and usefulness tests under result/stat_tests/.')
    display(bootstrap_df.head(20))
    display(usefulness_df.head(30))


## 10. Supplementary Cost and Configuration Diagnostics
The final section gathers supporting material that is useful for the report but should not obscure the main methodology:
- grouped table bundles,
- cost-drag summaries by asset, model, and strategy mode,
- best-configuration overview tables for quick inspection.


In [ ]:
if 'comparison' not in globals() or comparison is None or len(comparison) == 0:
    p = DATA_LOG_DIR / 'unified_experiment_summary.csv'
    comparison = pd.read_csv(p) if p.exists() else pd.DataFrame()
if 'plot_configs' not in globals() or plot_configs is None or len(plot_configs) == 0:
    plot_configs = _select_plot_configs(comparison) if len(comparison) > 0 else pd.DataFrame()

if len(comparison) == 0:
    print('No unified summary available for supplementary paper tables.')
    supplementary_tables = pd.DataFrame()
else:
    cost_drag = pd.read_csv(RESULT_DIR / 'paper_tables' / 'cost_comparison_by_configuration.csv') if (RESULT_DIR / 'paper_tables' / 'cost_comparison_by_configuration.csv').exists() else pd.DataFrame()
    if len(cost_drag) > 0:
        cost_drag_summary = (cost_drag.groupby(['Asset', 'Model', 'Strategy Mode'], as_index=False)
                             [['Sharpe Cost Drag', 'Avg Return Cost Drag', 'Cumulative Return Cost Drag']]
                             .mean()
                             .sort_values(['Asset', 'Strategy Mode', 'Sharpe Cost Drag'], ascending=[True, True, False]))
        cost_drag_summary.to_csv(RESULT_DIR / 'paper_tables' / 'cost_drag_summary_by_asset_model_strategy.csv', index=False)
        print('Average cost drag by asset, model, and strategy mode:')
        display(cost_drag_summary.head(20))
    else:
        cost_drag_summary = pd.DataFrame()

    bundle_frames = []
    for bundle_name in ['table10_bundle.csv', 'table11_bundle.csv', 'table12_bundle.csv']:
        bundle_path = RESULT_DIR / 'paper_tables' / bundle_name
        if bundle_path.exists():
            bundle_df = pd.read_csv(bundle_path)
            bundle_df.insert(0, 'Bundle', bundle_name.replace('_bundle.csv', ''))
            bundle_frames.append(bundle_df)

    supplementary_tables = pd.concat(bundle_frames, ignore_index=True) if len(bundle_frames) > 0 else pd.DataFrame()
    if len(supplementary_tables) > 0:
        supplementary_tables.to_csv(RESULT_DIR / 'paper_tables' / 'grouped_table_overview.csv', index=False)
        print('Grouped paper-style table overview saved under result/paper_tables/.')
        display(supplementary_tables.head(20))

    if len(plot_configs) > 0:
        best_cols = [
            'Asset', 'Model', 'Model Family', 'Feature Set', 'Strategy Mode', 'Cost Regime',
            'Validation Precision', 'Validation Recall', 'Validation Sharpe', 'Test Sharpe',
            'Test Precision', 'Test Recall', 'Test Avg Return', 'Threshold Feasible',
        ]
        best_config_overview = plot_configs.reindex(columns=best_cols).sort_values(
            ['Asset', 'Model', 'Strategy Mode', 'Cost Regime']
        ).reset_index(drop=True)
        best_config_overview.to_csv(RESULT_DIR / 'paper_tables' / 'best_config_overview.csv', index=False)
        print('Representative best-configuration overview saved under result/paper_tables/.')
        display(best_config_overview.head(20))
